# DC-Guardian Predictive Maintenance v2

## Stage 1 — Expanded Backblaze Data Inventory

The original Maintenance v1 model is preserved as the baseline.

Maintenance v2 first investigates a larger historical Backblaze dataset
for the target drive model ST12000NM0008.

No model architecture, feature engineering, threshold, or prediction
logic is changed during this stage.

Objectives:

1. Determine how much historical ST12000NM0008 data is available.
2. Count drive-day observations by period.
3. Count unique drives by period.
4. Count recorded failures and unique failed drives.
5. Select chronological train, validation, and test periods.
6. Re-run the existing v1 model methodology on the expanded dataset
   before introducing further enhancements.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# Reproducibility
# ============================================================

RANDOM_STATE = 42

np.random.seed(
    RANDOM_STATE
)


# ============================================================
# Locate DC-Guardian project
# ============================================================

cwd = Path.cwd().resolve()

print(
    "Current working directory:",
    cwd
)


# Notebook may be launched from:
#
# DC-Guardian/
#
# or:
#
# DC-Guardian/phase1/predictive_maintenance/notebooks/
#
# Handle both cases.

if cwd.name == "DC-Guardian":

    PROJECT_ROOT = cwd

elif cwd.name == "notebooks":

    PROJECT_ROOT = (
        cwd
        .parents[2]
    )

else:

    raise RuntimeError(
        "Run this notebook from either the "
        "DC-Guardian project root or the "
        "predictive_maintenance/notebooks folder."
    )


# ============================================================
# Maintenance paths
# ============================================================

MAINTENANCE_DIR = (
    PROJECT_ROOT
    / "phase1"
    / "predictive_maintenance"
)

RAW_DIR = (
    MAINTENANCE_DIR
    / "data"
    / "raw"
    / "backblaze"
)

PROCESSED_DIR = (
    MAINTENANCE_DIR
    / "data"
    / "processed"
)

MODEL_DIR = (
    MAINTENANCE_DIR
    / "models"
)


for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    MODEL_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# Model scope
# ============================================================

TARGET_MODEL = (
    "ST12000NM0008"
)

FAILURE_HORIZON_DAYS = 7


# ============================================================
# Verify configuration
# ============================================================

print(
    "\n============================================"
)

print(
    "DC-GUARDIAN MAINTENANCE V2"
)

print(
    "============================================"
)

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Maintenance directory:",
    MAINTENANCE_DIR
)

print(
    "Raw data directory:",
    RAW_DIR
)

print(
    "Processed directory:",
    PROCESSED_DIR
)

print(
    "Model directory:",
    MODEL_DIR
)

print(
    "Target drive model:",
    TARGET_MODEL
)

print(
    "Failure horizon:",
    FAILURE_HORIZON_DAYS,
    "days"
)

Current working directory: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\notebooks

DC-GUARDIAN MAINTENANCE V2
Project root: C:\Users\Aljawharah\Documents\DC-Guardian
Maintenance directory: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance
Raw data directory: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\raw\backblaze
Processed directory: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed
Model directory: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\models
Target drive model: ST12000NM0008
Failure horizon: 7 days


In [2]:
# ============================================================
# Recent Backblaze archive inventory
# ============================================================

import io
import zipfile


ARCHIVES = {
    "2025_Q2": RAW_DIR / "data_Q2_2025.zip",
    "2025_Q3": RAW_DIR / "data_Q3_2025.zip",
    "2025_Q4": RAW_DIR / "data_Q4_2025.zip",
    "2026_Q1": RAW_DIR / "data_Q1_2026.zip",
}


print(
    "\n============================================"
)
print(
    "BACKBLAZE ARCHIVE CHECK"
)
print(
    "============================================"
)


for period, archive_path in ARCHIVES.items():

    print(
        f"{period}: "
        f"{'FOUND' if archive_path.exists() else 'MISSING'}"
    )

    print(
        f"  {archive_path}"
    )


missing_archives = [
    str(path)
    for path in ARCHIVES.values()
    if not path.exists()
]


if missing_archives:

    raise FileNotFoundError(
        "Missing Backblaze archives:\n"
        + "\n".join(
            missing_archives
        )
    )


print(
    "\nPASS: All four Backblaze archives found."
)


BACKBLAZE ARCHIVE CHECK
2025_Q2: FOUND
  C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\raw\backblaze\data_Q2_2025.zip
2025_Q3: FOUND
  C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\raw\backblaze\data_Q3_2025.zip
2025_Q4: FOUND
  C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\raw\backblaze\data_Q4_2025.zip
2026_Q1: FOUND
  C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\raw\backblaze\data_Q1_2026.zip

PASS: All four Backblaze archives found.


In [3]:
# ============================================================
# Recent Backblaze archive inventory
# ============================================================

import io
import zipfile


ARCHIVES = {
    "2025_Q2": RAW_DIR / "data_Q2_2025.zip",
    "2025_Q3": RAW_DIR / "data_Q3_2025.zip",
    "2025_Q4": RAW_DIR / "data_Q4_2025.zip",
    "2026_Q1": RAW_DIR / "data_Q1_2026.zip",
}


print(
    "\n============================================"
)
print(
    "BACKBLAZE ARCHIVE CHECK"
)
print(
    "============================================"
)


for period, archive_path in ARCHIVES.items():

    print(
        f"{period}: "
        f"{'FOUND' if archive_path.exists() else 'MISSING'}"
    )

    print(
        f"  {archive_path}"
    )


missing_archives = [
    str(path)
    for path in ARCHIVES.values()
    if not path.exists()
]


if missing_archives:

    raise FileNotFoundError(
        "Missing Backblaze archives:\n"
        + "\n".join(
            missing_archives
        )
    )


print(
    "\nPASS: All four Backblaze archives found."
)


BACKBLAZE ARCHIVE CHECK
2025_Q2: FOUND
  C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\raw\backblaze\data_Q2_2025.zip
2025_Q3: FOUND
  C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\raw\backblaze\data_Q3_2025.zip
2025_Q4: FOUND
  C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\raw\backblaze\data_Q4_2025.zip
2026_Q1: FOUND
  C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\raw\backblaze\data_Q1_2026.zip

PASS: All four Backblaze archives found.


In [11]:
# ============================================================
# Inventory target model directly from ZIP archives
# ============================================================

INVENTORY_COLUMNS = [
    "date",
    "serial_number",
    "model",
    "failure",
]


inventory_results = []


for period, archive_path in ARCHIVES.items():

    print(
        "\n============================================"
    )
    print(
        f"PROCESSING {period}"
    )
    print(
        "============================================"
    )

    period_rows = 0

    period_serials = set()

    period_failure_rows = 0

    period_failed_serials = set()

    files_processed = 0


    with zipfile.ZipFile(
        archive_path,
        "r"
    ) as archive:

        csv_files = [
            name
            for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]


        print(
            "CSV files found:",
            len(csv_files)
        )


        for file_number, csv_name in enumerate(
            csv_files,
            start=1
        ):

            with archive.open(
                csv_name
            ) as file:

                day = pd.read_csv(
                    file,
                    usecols=lambda column:
                        column
                        in INVENTORY_COLUMNS,
                    low_memory=False
                )


            # ------------------------------------------------
            # Required-column validation
            # ------------------------------------------------

            required = {
                "date",
                "serial_number",
                "model",
                "failure",
            }

            missing = (
                required
                - set(day.columns)
            )


            if missing:

                raise ValueError(
                    f"{period} / {csv_name} "
                    f"is missing columns: "
                    f"{sorted(missing)}"
                )


            # ------------------------------------------------
            # Keep target model only
            # ------------------------------------------------

            target_day = day[
                day["model"]
                .eq(TARGET_MODEL)
            ]


            if not target_day.empty:

                period_rows += len(
                    target_day
                )


                period_serials.update(
                    target_day[
                        "serial_number"
                    ]
                    .dropna()
                    .astype(str)
                )


                failed = target_day[
                    target_day[
                        "failure"
                    ].eq(1)
                ]


                period_failure_rows += len(
                    failed
                )


                period_failed_serials.update(
                    failed[
                        "serial_number"
                    ]
                    .dropna()
                    .astype(str)
                )


            files_processed += 1


            # Progress every 15 files
            if (
                file_number % 15 == 0
                or
                file_number == len(csv_files)
            ):

                print(
                    f"  {file_number:>3}/"
                    f"{len(csv_files)} files | "
                    f"target rows="
                    f"{period_rows:,} | "
                    f"failures="
                    f"{period_failure_rows:,}"
                )


    inventory_results.append(
        {
            "period":
                period,

            "files":
                files_processed,

            "drive_days":
                period_rows,

            "unique_drives":
                len(
                    period_serials
                ),

            "failure_rows":
                period_failure_rows,

            "failed_drives":
                len(
                    period_failed_serials
                ),
        }
    )


# ============================================================
# Inventory table
# ============================================================

recent_inventory = pd.DataFrame(
    inventory_results
)


print(
    "\n============================================"
)
print(
    f"{TARGET_MODEL} RECENT DATA INVENTORY"
)
print(
    "============================================"
)


display(
    recent_inventory
)


PROCESSING 2025_Q2
CSV files found: 91
   15/91 files | target rows=285,100 | failures=24
   30/91 files | target rows=569,570 | failures=43
   45/91 files | target rows=848,591 | failures=70
   60/91 files | target rows=1,132,746 | failures=90
   75/91 files | target rows=1,416,180 | failures=112
   90/91 files | target rows=1,699,405 | failures=137
   91/91 files | target rows=1,718,302 | failures=138

PROCESSING 2025_Q3
CSV files found: 92
   15/92 files | target rows=282,848 | failures=18
   30/92 files | target rows=565,117 | failures=32
   45/92 files | target rows=847,807 | failures=57
   60/92 files | target rows=1,129,944 | failures=76
   75/92 files | target rows=1,410,329 | failures=105
   90/92 files | target rows=1,691,247 | failures=127
   92/92 files | target rows=1,728,706 | failures=132

PROCESSING 2025_Q4
CSV files found: 92
   15/92 files | target rows=280,681 | failures=28
   30/92 files | target rows=561,367 | failures=50
   45/92 files | target rows=841,325 | fai

,period,files,drive_days,unique_drives,failure_rows,failed_drives
0,2025_Q2,91,1718302,19041,138,138
1,2025_Q3,92,1728706,18902,132,132
2,2025_Q4,92,1715709,18771,112,112
3,2026_Q1,90,1664558,18650,129,129


In [12]:
inventory_output = (
    INVENTORY_DIR
    / "ST12000NM0008_2025Q2_2026Q1_inventory.csv"
)

recent_inventory.to_csv(
    inventory_output,
    index=False
)

print(
    "Saved inventory:",
    inventory_output
)

Saved inventory: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\inventory\ST12000NM0008_2025Q2_2026Q1_inventory.csv


In [13]:
# ============================================================
# Inventory ALL drive models across recent Backblaze archives
# ============================================================

all_model_parts = []

for period, archive_path in ARCHIVES.items():

    print(
        "\n============================================"
    )
    print(
        f"INVENTORYING ALL MODELS: {period}"
    )
    print(
        "============================================"
    )

    period_parts = []

    with zipfile.ZipFile(
        archive_path,
        "r"
    ) as archive:

        csv_files = [
            name
            for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]

        for file_number, csv_name in enumerate(
            csv_files,
            start=1
        ):

            with archive.open(csv_name) as file:

                day = pd.read_csv(
                    file,
                    usecols=[
                        "date",
                        "serial_number",
                        "model",
                        "failure",
                    ],
                    low_memory=False
                )

            period_parts.append(
                day
            )

            if (
                file_number % 15 == 0
                or file_number == len(csv_files)
            ):

                print(
                    f"  Processed "
                    f"{file_number}/"
                    f"{len(csv_files)} files"
                )

    period_df = pd.concat(
        period_parts,
        ignore_index=True
    )

    # --------------------------------------------------------
    # Summary per model for this period
    # --------------------------------------------------------

    summary = (
        period_df
        .groupby("model")
        .agg(
            drive_days=(
                "serial_number",
                "size"
            ),

            unique_drives=(
                "serial_number",
                "nunique"
            ),

            failure_rows=(
                "failure",
                "sum"
            ),
        )
        .reset_index()
    )

    failed_drives = (
        period_df[
            period_df["failure"].eq(1)
        ]
        .groupby("model")[
            "serial_number"
        ]
        .nunique()
        .rename("failed_drives")
        .reset_index()
    )

    summary = summary.merge(
        failed_drives,
        on="model",
        how="left"
    )

    summary[
        "failed_drives"
    ] = (
        summary["failed_drives"]
        .fillna(0)
        .astype(int)
    )

    summary[
        "failure_rows"
    ] = (
        summary["failure_rows"]
        .fillna(0)
        .astype(int)
    )

    summary.insert(
        0,
        "period",
        period
    )

    all_model_parts.append(
        summary
    )


# ============================================================
# Combine period inventories
# ============================================================

all_model_inventory = pd.concat(
    all_model_parts,
    ignore_index=True
)


print(
    "\n============================================"
)
print(
    "ALL-MODEL INVENTORY COMPLETE"
)
print(
    "============================================"
)

print(
    "Models:",
    all_model_inventory[
        "model"
    ].nunique()
)

display(
    all_model_inventory.head()
)


INVENTORYING ALL MODELS: 2025_Q2
  Processed 15/91 files
  Processed 30/91 files
  Processed 45/91 files
  Processed 60/91 files
  Processed 75/91 files
  Processed 90/91 files
  Processed 91/91 files

INVENTORYING ALL MODELS: 2025_Q3
  Processed 15/92 files
  Processed 30/92 files
  Processed 45/92 files
  Processed 60/92 files
  Processed 75/92 files
  Processed 90/92 files
  Processed 92/92 files

INVENTORYING ALL MODELS: 2025_Q4
  Processed 15/92 files
  Processed 30/92 files
  Processed 45/92 files
  Processed 60/92 files
  Processed 75/92 files
  Processed 90/92 files
  Processed 92/92 files

INVENTORYING ALL MODELS: 2026_Q1
  Processed 15/90 files
  Processed 30/90 files
  Processed 45/90 files
  Processed 60/90 files
  Processed 75/90 files
  Processed 90/90 files

ALL-MODEL INVENTORY COMPLETE
Models: 80


,period,model,drive_days,unique_drives,failure_rows,failed_drives
0,2025_Q2,CT250MX500SSD1,63750,709,0,0
1,2025_Q2,DELLBOSS VD,39212,432,0,0
2,2025_Q2,HGST HMS5C4040ALE640,990,11,0,0
3,2025_Q2,HGST HMS5C4040BLE640,16893,188,1,1
4,2025_Q2,HGST HUH721010ALE600,1818,20,0,0


In [14]:
# ============================================================
# Candidate drive-model analysis
# ============================================================

candidate_summary = (
    all_model_inventory
    .groupby("model")
    .agg(
        total_drive_days=(
            "drive_days",
            "sum"
        ),

        total_failure_rows=(
            "failure_rows",
            "sum"
        ),

        periods_present=(
            "period",
            "nunique"
        ),

        periods_with_failures=(
            "failed_drives",
            lambda x: (x > 0).sum()
        ),
    )
    .reset_index()
)


# ============================================================
# Add period-specific failed-drive counts
# ============================================================

failure_pivot = (
    all_model_inventory
    .pivot_table(
        index="model",
        columns="period",
        values="failed_drives",
        fill_value=0
    )
    .reset_index()
)


candidate_summary = (
    candidate_summary
    .merge(
        failure_pivot,
        on="model",
        how="left"
    )
)


# ============================================================
# Initial suitability filter
# ============================================================
#
# For now:
#
# - model must exist in all four periods
# - model must have failures in training
# - model must have failures in validation
# - model must have failures in final test
#
# These are candidate-selection rules only.
# We can tighten them after inspecting the results.
# ============================================================

candidate_summary[
    "train_failures"
] = (
    candidate_summary.get(
        "2025_Q2",
        0
    )
    +
    candidate_summary.get(
        "2025_Q3",
        0
    )
)


candidate_summary[
    "validation_failures"
] = candidate_summary.get(
    "2025_Q4",
    0
)


candidate_summary[
    "test_failures"
] = candidate_summary.get(
    "2026_Q1",
    0
)


candidate_summary[
    "eligible"
] = (
    candidate_summary[
        "periods_present"
    ].eq(4)
    &
    candidate_summary[
        "train_failures"
    ].gt(0)
    &
    candidate_summary[
        "validation_failures"
    ].gt(0)
    &
    candidate_summary[
        "test_failures"
    ].gt(0)
)


candidate_summary = (
    candidate_summary
    .sort_values(
        [
            "eligible",
            "total_failure_rows",
            "total_drive_days"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n============================================"
)
print(
    "MULTI-MODEL CANDIDATES"
)
print(
    "============================================"
)

display(
    candidate_summary[
        [
            "model",
            "periods_present",
            "total_drive_days",
            "train_failures",
            "validation_failures",
            "test_failures",
            "total_failure_rows",
            "eligible",
        ]
    ]
    .head(30)
)


MULTI-MODEL CANDIDATES


,model,periods_present,total_drive_days,train_failures,validation_failures,test_failures,total_failure_rows,eligible
0,ST12000NM0008,4,6827275,270.0,112.0,129.0,511,True
1,TOSHIBA MG07ACA14TA,4,13612521,223.0,76.0,94.0,393,True
2,TOSHIBA MG08ACA16TA,4,14555837,200.0,68.0,102.0,370,True
3,HGST HUH721212ALN604,4,3587124,193.0,76.0,95.0,364,True
4,HGST HUH721212ALE604,4,4839627,180.0,85.0,86.0,351,True
5,TOSHIBA MG08ACA16TEY,4,1814543,248.0,51.0,26.0,325,True
6,ST8000NM0055,4,4834057,126.0,56.0,39.0,221,True
7,ST24000NM002H,4,2530694,74.0,65.0,74.0,213,True
8,WDC WUH721816ALE6L4,4,9605738,95.0,44.0,56.0,195,True
9,ST16000NM001G,4,12450309,106.0,35.0,44.0,185,True


In [15]:
all_model_inventory.to_csv(
    INVENTORY_DIR
    / "all_models_2025Q2_2026Q1_inventory.csv",
    index=False
)

candidate_summary.to_csv(
    INVENTORY_DIR
    / "multi_model_candidates.csv",
    index=False
)

print(
    "PASS: Multi-model inventories saved."
)

PASS: Multi-model inventories saved.


In [7]:
# ============================================================
# Initial multi-model candidate set
# ============================================================

CANDIDATE_MODELS = [
    "ST12000NM0008",
    "TOSHIBA MG07ACA14TA",
    "TOSHIBA MG08ACA16TA",
    "HGST HUH721212ALN604",
    "HGST HUH721212ALE604",
    "TOSHIBA MG08ACA16TEY",
    "ST8000NM0055",
    "ST24000NM002H",
    "WDC WUH721816ALE6L4",
    "ST16000NM001G",
]


print(
    "Candidate models:",
    len(CANDIDATE_MODELS)
)

for model in CANDIDATE_MODELS:
    print(
        " -",
        model
    )

Candidate models: 10
 - ST12000NM0008
 - TOSHIBA MG07ACA14TA
 - TOSHIBA MG08ACA16TA
 - HGST HUH721212ALN604
 - HGST HUH721212ALE604
 - TOSHIBA MG08ACA16TEY
 - ST8000NM0055
 - ST24000NM002H
 - WDC WUH721816ALE6L4
 - ST16000NM001G


In [17]:
# ============================================================
# SMART feature coverage analysis
# ============================================================

from collections import defaultdict


# ------------------------------------------------------------
# Discover SMART columns from one daily file
# ------------------------------------------------------------

first_archive_path = next(
    iter(ARCHIVES.values())
)

with zipfile.ZipFile(
    first_archive_path,
    "r"
) as archive:

    first_csv = next(
        name
        for name in archive.namelist()
        if name.lower().endswith(".csv")
    )

    with archive.open(
        first_csv
    ) as file:

        sample_columns = pd.read_csv(
            file,
            nrows=1
        ).columns.tolist()


SMART_COLUMNS = [
    column
    for column in sample_columns
    if column.startswith("smart_")
]


print(
    "SMART columns discovered:",
    len(SMART_COLUMNS)
)


# ============================================================
# Coverage counters
# ============================================================

row_counts = defaultdict(
    int
)

non_null_counts = defaultdict(
    lambda: defaultdict(int)
)


# ============================================================
# Scan archives
# ============================================================

for period, archive_path in ARCHIVES.items():

    print(
        "\n============================================"
    )
    print(
        f"SMART COVERAGE: {period}"
    )
    print(
        "============================================"
    )


    with zipfile.ZipFile(
        archive_path,
        "r"
    ) as archive:

        csv_files = [
            name
            for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]


        for file_number, csv_name in enumerate(
            csv_files,
            start=1
        ):

            with archive.open(
                csv_name
            ) as file:

                day = pd.read_csv(
                    file,
                    usecols=(
                        [
                            "model"
                        ]
                        + SMART_COLUMNS
                    ),
                    low_memory=False
                )


            # ------------------------------------------------
            # Candidate models only
            # ------------------------------------------------

            day = day[
                day["model"].isin(
                    CANDIDATE_MODELS
                )
            ]


            if day.empty:
                continue


            # ------------------------------------------------
            # Calculate model-level coverage
            # ------------------------------------------------

            for model, model_data in (
                day.groupby("model")
            ):

                row_counts[
                    (period, model)
                ] += len(
                    model_data
                )


                counts = (
                    model_data[
                        SMART_COLUMNS
                    ]
                    .notna()
                    .sum()
                )


                for column, count in (
                    counts.items()
                ):

                    non_null_counts[
                        (period, model)
                    ][column] += int(
                        count
                    )


            if (
                file_number % 15 == 0
                or
                file_number == len(
                    csv_files
                )
            ):

                print(
                    f"  Processed "
                    f"{file_number}/"
                    f"{len(csv_files)} files"
                )


print(
    "\nPASS: SMART coverage scan complete."
)

SMART columns discovered: 186

SMART COVERAGE: 2025_Q2
  Processed 15/91 files
  Processed 30/91 files
  Processed 45/91 files
  Processed 60/91 files
  Processed 75/91 files
  Processed 90/91 files
  Processed 91/91 files

SMART COVERAGE: 2025_Q3
  Processed 15/92 files
  Processed 30/92 files
  Processed 45/92 files
  Processed 60/92 files
  Processed 75/92 files
  Processed 90/92 files
  Processed 92/92 files

SMART COVERAGE: 2025_Q4
  Processed 15/92 files
  Processed 30/92 files
  Processed 45/92 files
  Processed 60/92 files
  Processed 75/92 files
  Processed 90/92 files
  Processed 92/92 files

SMART COVERAGE: 2026_Q1
  Processed 15/90 files
  Processed 30/90 files
  Processed 45/90 files
  Processed 60/90 files
  Processed 75/90 files
  Processed 90/90 files

PASS: SMART coverage scan complete.


In [21]:
# ============================================================
# Build SMART coverage table
# ============================================================

coverage_rows = []


for (
    period,
    model
), total_rows in row_counts.items():

    row = {
        "period":
            period,

        "model":
            model,

        "rows":
            total_rows,
    }


    for smart_column in SMART_COLUMNS:

        non_null = (
            non_null_counts[
                (period, model)
            ].get(
                smart_column,
                0
            )
        )


        row[
            smart_column
        ] = (
            non_null
            / total_rows
            * 100
        )


    coverage_rows.append(
        row
    )


smart_coverage = pd.DataFrame(
    coverage_rows
)


print(
    "Coverage rows:",
    len(smart_coverage)
)

display(
    smart_coverage.head()
)

Coverage rows: 40


,period,model,rows,smart_1_normalized,smart_1_raw,smart_2_normalized,smart_2_raw,smart_3_normalized,smart_3_raw,smart_4_normalized,...,smart_250_normalized,smart_250_raw,smart_251_normalized,smart_251_raw,smart_252_normalized,smart_252_raw,smart_254_normalized,smart_254_raw,smart_255_normalized,smart_255_raw
0,2025_Q2,HGST HUH721212ALE604,1207348,99.999917,99.999917,99.999917,99.999917,99.999917,99.999917,99.999917,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025_Q2,HGST HUH721212ALN604,903548,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025_Q2,ST12000NM0008,1718302,99.999884,99.999884,0.000000,0.000000,99.999884,99.999884,99.999884,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025_Q2,ST16000NM001G,3078040,99.999643,99.999643,0.000000,0.000000,99.999643,99.999643,99.999643,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025_Q2,ST24000NM002H,432651,99.999769,99.999769,0.000000,0.000000,99.999769,99.999769,99.999769,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
# ============================================================
# Build SMART feature compatibility summary
# ============================================================

coverage_columns = [
    column
    for column in smart_coverage.columns
    if column.startswith("smart_")
]


coverage_summary = pd.DataFrame(
    {
        "smart_feature":
            coverage_columns,

        "minimum_coverage_pct": [
            smart_coverage[column].min()
            for column in coverage_columns
        ],

        "mean_coverage_pct": [
            smart_coverage[column].mean()
            for column in coverage_columns
        ],
    }
)


coverage_summary = (
    coverage_summary
    .sort_values(
        [
            "minimum_coverage_pct",
            "mean_coverage_pct"
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n============================================"
)
print(
    "SMART FEATURE COMPATIBILITY"
)
print(
    "============================================"
)

print(
    "SMART features analyzed:",
    len(coverage_summary)
)


display(
    coverage_summary.head(40)
)


SMART FEATURE COMPATIBILITY
SMART features analyzed: 186


,smart_feature,minimum_coverage_pct,mean_coverage_pct
0,smart_1_normalized,99.921472,99.997569
1,smart_1_raw,99.921472,99.997569
2,smart_3_normalized,99.921472,99.997569
3,smart_3_raw,99.921472,99.997569
4,smart_4_normalized,99.921472,99.997569
5,smart_4_raw,99.921472,99.997569
6,smart_5_normalized,99.921472,99.997569
7,smart_5_raw,99.921472,99.997569
8,smart_7_normalized,99.921472,99.997569
9,smart_7_raw,99.921472,99.997569


In [24]:
# ============================================================
# Identify common SMART features
# ============================================================

COMMON_COVERAGE_THRESHOLD = 95.0


common_smart_features = (
    coverage_summary[
        coverage_summary[
            "minimum_coverage_pct"
        ]
        >= COMMON_COVERAGE_THRESHOLD
    ][
        "smart_feature"
    ]
    .tolist()
)


print(
    "\n============================================"
)
print(
    "COMMON SMART FEATURES"
)
print(
    "============================================"
)

print(
    "Coverage threshold:",
    f"{COMMON_COVERAGE_THRESHOLD:.0f}%"
)

print(
    "Common features:",
    len(common_smart_features)
)


for feature in common_smart_features:

    print(
        feature
    )


COMMON SMART FEATURES
Coverage threshold: 95%
Common features: 26
smart_1_normalized
smart_1_raw
smart_3_normalized
smart_3_raw
smart_4_normalized
smart_4_raw
smart_5_normalized
smart_5_raw
smart_7_normalized
smart_7_raw
smart_9_normalized
smart_9_raw
smart_10_normalized
smart_10_raw
smart_12_normalized
smart_12_raw
smart_192_normalized
smart_192_raw
smart_193_normalized
smart_193_raw
smart_194_normalized
smart_194_raw
smart_198_normalized
smart_198_raw
smart_199_normalized
smart_199_raw


In [25]:
# ============================================================
# Separate raw and normalized common SMART features
# ============================================================

common_raw_features = [
    feature
    for feature in common_smart_features
    if feature.endswith("_raw")
]

common_normalized_features = [
    feature
    for feature in common_smart_features
    if feature.endswith("_normalized")
]


print(
    "\n============================================"
)
print(
    "COMMON SMART FEATURE TYPES"
)
print(
    "============================================"
)

print(
    "Raw features:",
    len(common_raw_features)
)

for feature in common_raw_features:
    print(" ", feature)


print(
    "\nNormalized features:",
    len(common_normalized_features)
)

for feature in common_normalized_features:
    print(" ", feature)


COMMON SMART FEATURE TYPES
Raw features: 13
  smart_1_raw
  smart_3_raw
  smart_4_raw
  smart_5_raw
  smart_7_raw
  smart_9_raw
  smart_10_raw
  smart_12_raw
  smart_192_raw
  smart_193_raw
  smart_194_raw
  smart_198_raw
  smart_199_raw

Normalized features: 13
  smart_1_normalized
  smart_3_normalized
  smart_4_normalized
  smart_5_normalized
  smart_7_normalized
  smart_9_normalized
  smart_10_normalized
  smart_12_normalized
  smart_192_normalized
  smart_193_normalized
  smart_194_normalized
  smart_198_normalized
  smart_199_normalized


In [26]:
# ============================================================
# Common raw SMART feature statistics
# ============================================================

smart_stats_parts = []


READ_COLUMNS = [
    "model",
] + common_raw_features


for period, archive_path in ARCHIVES.items():

    print(
        "\n============================================"
    )
    print(
        f"SMART VARIANCE ANALYSIS: {period}"
    )
    print(
        "============================================"
    )


    with zipfile.ZipFile(
        archive_path,
        "r"
    ) as archive:

        csv_files = [
            name
            for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]


        period_parts = []


        for file_number, csv_name in enumerate(
            csv_files,
            start=1
        ):

            with archive.open(csv_name) as file:

                day = pd.read_csv(
                    file,
                    usecols=lambda column:
                        column in READ_COLUMNS,
                    low_memory=False
                )


            day = day[
                day["model"].isin(
                    CANDIDATE_MODELS
                )
            ]


            if not day.empty:

                period_parts.append(
                    day
                )


            if (
                file_number % 15 == 0
                or
                file_number == len(csv_files)
            ):

                print(
                    f"  Processed "
                    f"{file_number}/"
                    f"{len(csv_files)} files"
                )


        period_df = pd.concat(
            period_parts,
            ignore_index=True
        )


        # ----------------------------------------------------
        # Statistics by model
        # ----------------------------------------------------

        for model, model_df in (
            period_df.groupby("model")
        ):

            for feature in common_raw_features:

                values = pd.to_numeric(
                    model_df[feature],
                    errors="coerce"
                )


                smart_stats_parts.append(
                    {
                        "period":
                            period,

                        "model":
                            model,

                        "feature":
                            feature,

                        "rows":
                            len(values),

                        "non_null":
                            int(
                                values.notna().sum()
                            ),

                        "unique_values":
                            int(
                                values.nunique(
                                    dropna=True
                                )
                            ),

                        "mean":
                            values.mean(),

                        "std":
                            values.std(),

                        "min":
                            values.min(),

                        "max":
                            values.max(),
                    }
                )


smart_feature_stats = pd.DataFrame(
    smart_stats_parts
)


print(
    "\nPASS: SMART variance analysis complete."
)


SMART VARIANCE ANALYSIS: 2025_Q2
  Processed 15/91 files
  Processed 30/91 files
  Processed 45/91 files
  Processed 60/91 files
  Processed 75/91 files
  Processed 90/91 files
  Processed 91/91 files

SMART VARIANCE ANALYSIS: 2025_Q3
  Processed 15/92 files
  Processed 30/92 files
  Processed 45/92 files
  Processed 60/92 files
  Processed 75/92 files
  Processed 90/92 files
  Processed 92/92 files

SMART VARIANCE ANALYSIS: 2025_Q4
  Processed 15/92 files
  Processed 30/92 files
  Processed 45/92 files
  Processed 60/92 files
  Processed 75/92 files
  Processed 90/92 files
  Processed 92/92 files

SMART VARIANCE ANALYSIS: 2026_Q1
  Processed 15/90 files
  Processed 30/90 files
  Processed 45/90 files
  Processed 60/90 files
  Processed 75/90 files
  Processed 90/90 files

PASS: SMART variance analysis complete.


In [27]:
# ============================================================
# Summarize raw SMART feature usefulness
# ============================================================

raw_feature_summary = (
    smart_feature_stats
    .groupby("feature")
    .agg(
        minimum_unique_values=(
            "unique_values",
            "min"
        ),

        median_unique_values=(
            "unique_values",
            "median"
        ),

        minimum_std=(
            "std",
            "min"
        ),

        median_std=(
            "std",
            "median"
        ),

        maximum_value=(
            "max",
            "max"
        ),
    )
    .reset_index()
)


raw_feature_summary[
    "constant_somewhere"
] = (
    raw_feature_summary[
        "minimum_unique_values"
    ]
    <= 1
)


raw_feature_summary = (
    raw_feature_summary
    .sort_values(
        [
            "constant_somewhere",
            "median_unique_values"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n============================================"
)
print(
    "COMMON RAW SMART FEATURE SUMMARY"
)
print(
    "============================================"
)


display(
    raw_feature_summary
)


COMMON RAW SMART FEATURE SUMMARY


,feature,minimum_unique_values,median_unique_values,minimum_std,median_std,maximum_value,constant_somewhere
0,smart_9_raw,4777,35192.0,1200.515668,5.982324e+03,7.818700e+04,False
1,smart_193_raw,358,4173.0,118.571814,9.833277e+02,2.761320e+07,False
2,smart_192_raw,21,521.0,3.842061,6.623804e+01,2.761118e+07,False
3,smart_5_raw,41,453.0,12.254188,1.434469e+02,6.552800e+04,False
4,smart_198_raw,2,145.5,0.006167,3.253336e+01,4.617600e+05,False
5,smart_4_raw,23,110.0,4.099257,2.717585e+01,5.422000e+03,False
6,smart_12_raw,22,99.5,4.063024,2.712955e+01,4.596000e+03,False
7,smart_194_raw,25,41.0,3.589291,4.513210e+00,1.010000e+02,False
8,smart_1_raw,1,1628.5,0.000000,2.394428e+06,2.837516e+09,True
9,smart_199_raw,1,169.0,0.000000,4.420515e+01,5.095000e+04,True


In [5]:
# ============================================================
# Maintenance v2 common SMART candidates
# ============================================================

BASE_SMART_FEATURES = [
    "smart_5_raw",
    "smart_9_raw",
    "smart_192_raw",
    "smart_193_raw",
    "smart_194_raw",
    "smart_198_raw",
    "smart_4_raw",
    "smart_12_raw",
]


print(
    "\n============================================"
)
print(
    "MAINTENANCE V2 BASE SMART FEATURES"
)
print(
    "============================================"
)

for feature in BASE_SMART_FEATURES:
    print(feature)


MAINTENANCE V2 BASE SMART FEATURES
smart_5_raw
smart_9_raw
smart_192_raw
smart_193_raw
smart_194_raw
smart_198_raw
smart_4_raw
smart_12_raw


In [30]:
%pip install pyarrow

   ---------------------------------------- 0.0/28.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/28.6 MB ? eta -:--:--
   - -------------------------------------- 0.8/28.6 MB 4.5 MB/s eta 0:00:07
   ----- ---------------------------------- 3.9/28.6 MB 11.7 MB/s eta 0:00:03
   --------------------- ------------------ 15.7/28.6 MB 29.7 MB/s eta 0:00:01
   ---------------------- ----------------- 16.3/28.6 MB 25.4 MB/s eta 0:00:01
   ----------------------- ---------------- 17.0/28.6 MB 18.7 MB/s eta 0:00:01
   -------------------------- ------------- 19.1/28.6 MB 16.2 MB/s eta 0:00:01
   ------------------------------ --------- 22.0/28.6 MB 16.2 MB/s eta 0:00:01
   ---------------------------------- ----- 24.4/28.6 MB 15.9 MB/s eta 0:00:01
   ------------------------------------ --- 26.2/28.6 MB 15.1 MB/s eta 0:00:01
   ---------------------------------------- 28.6/28.6 MB 14.5 MB/s  0:00:02
Note: you may need to restart the kernel to use updated packages.


In [8]:
# ============================================================
# Extract compact Maintenance v2 datasets
# ============================================================

COMPACT_DIR = (
    PROCESSED_DIR
    / "maintenance_v2_compact"
)

COMPACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


COMPACT_COLUMNS = [
    "date",
    "serial_number",
    "model",
    "capacity_bytes",
    "failure",
] + BASE_SMART_FEATURES


compact_outputs = {}


for period, archive_path in ARCHIVES.items():

    print(
        "\n============================================"
    )
    print(
        f"EXTRACTING COMPACT DATA: {period}"
    )
    print(
        "============================================"
    )

    parts = []


    with zipfile.ZipFile(
        archive_path,
        "r"
    ) as archive:

        csv_files = [
            name
            for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]


        for file_number, csv_name in enumerate(
            csv_files,
            start=1
        ):

            with archive.open(
                csv_name
            ) as file:

                day = pd.read_csv(
                    file,
                    usecols=lambda column:
                        column in COMPACT_COLUMNS,
                    low_memory=False
                )


            # ------------------------------------------------
            # Keep selected candidate drive models only
            # ------------------------------------------------

            day = day[
                day["model"].isin(
                    CANDIDATE_MODELS
                )
            ].copy()


            if not day.empty:

                parts.append(
                    day
                )


            if (
                file_number % 15 == 0
                or
                file_number == len(csv_files)
            ):

                print(
                    f"  Processed "
                    f"{file_number}/"
                    f"{len(csv_files)} files"
                )


    # ========================================================
    # Combine quarter
    # ========================================================

    quarter_df = pd.concat(
        parts,
        ignore_index=True
    )


    quarter_df[
        "date"
    ] = pd.to_datetime(
        quarter_df["date"],
        errors="raise"
    )


    quarter_df[
        "failure"
    ] = pd.to_numeric(
        quarter_df["failure"],
        errors="raise"
    ).astype(
        "int8"
    )


    # --------------------------------------------------------
    # Sort drive histories
    # --------------------------------------------------------

    quarter_df = (
        quarter_df
        .sort_values(
            [
                "serial_number",
                "date"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Save compact Parquet
    # --------------------------------------------------------

    output_file = (
        COMPACT_DIR
        / f"{period}_candidate_drives.parquet"
    )


    quarter_df.to_parquet(
        output_file,
        index=False
    )


    compact_outputs[
        period
    ] = output_file


    print(
        "Rows:",
        f"{len(quarter_df):,}"
    )

    print(
        "Drives:",
        f"{quarter_df['serial_number'].nunique():,}"
    )

    print(
        "Failures:",
        f"{int(quarter_df['failure'].sum()):,}"
    )

    print(
        "Saved:",
        output_file
    )


    # Important:
    # free the large quarter before reading the next one.

    del quarter_df
    del parts


EXTRACTING COMPACT DATA: 2025_Q2
  Processed 15/91 files
  Processed 30/91 files
  Processed 45/91 files
  Processed 60/91 files
  Processed 75/91 files
  Processed 90/91 files
  Processed 91/91 files
Rows: 18,465,014
Drives: 205,381
Failures: 773
Saved: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_compact\2025_Q2_candidate_drives.parquet

EXTRACTING COMPACT DATA: 2025_Q3
  Processed 15/92 files
  Processed 30/92 files
  Processed 45/92 files
  Processed 60/92 files
  Processed 75/92 files
  Processed 90/92 files
  Processed 92/92 files
Rows: 18,833,929
Drives: 206,234
Failures: 942
Saved: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_compact\2025_Q3_candidate_drives.parquet

EXTRACTING COMPACT DATA: 2025_Q4
  Processed 15/92 files
  Processed 30/92 files
  Processed 45/92 files
  Processed 60/92 files
  Processed 75/92 files
  Processed 90/92 files
  Processed 92/92 files


In [9]:
# ============================================================
# Verify compact Maintenance v2 datasets
# ============================================================

compact_summary = []


for period in ARCHIVES:

    file_path = (
        COMPACT_DIR
        / f"{period}_candidate_drives.parquet"
    )


    if not file_path.exists():

        raise FileNotFoundError(
            f"Missing compact dataset: "
            f"{file_path}"
        )


    quarter = pd.read_parquet(
        file_path,
        columns=[
            "date",
            "serial_number",
            "model",
            "failure",
        ]
    )


    compact_summary.append(
        {
            "period":
                period,

            "drive_days":
                len(quarter),

            "unique_drives":
                quarter[
                    "serial_number"
                ].nunique(),

            "models":
                quarter[
                    "model"
                ].nunique(),

            "failure_rows":
                int(
                    quarter[
                        "failure"
                    ].sum()
                ),

            "start_date":
                quarter[
                    "date"
                ].min(),

            "end_date":
                quarter[
                    "date"
                ].max(),

            "file_size_gb":
                file_path.stat().st_size
                / (1024 ** 3),
        }
    )


    del quarter


compact_summary = pd.DataFrame(
    compact_summary
)


print(
    "\n============================================"
)
print(
    "MAINTENANCE V2 COMPACT DATA SUMMARY"
)
print(
    "============================================"
)


display(
    compact_summary
)


MAINTENANCE V2 COMPACT DATA SUMMARY


,period,drive_days,unique_drives,models,failure_rows,start_date,end_date,file_size_gb
0,2025_Q2,18465014,205381,10,773,2025-04-01,2025-06-30,0.071036
1,2025_Q3,18833929,206234,10,942,2025-07-01,2025-09-30,0.072505
2,2025_Q4,18852311,206957,10,668,2025-10-01,2025-12-31,0.072226
3,2026_Q1,18506471,207782,10,745,2026-01-01,2026-03-31,0.072220


In [10]:
print(
    "Total drive-days:",
    f"{compact_summary['drive_days'].sum():,}"
)

print(
    "Total failure rows:",
    f"{compact_summary['failure_rows'].sum():,}"
)

print(
    "Total Parquet size:",
    f"{compact_summary['file_size_gb'].sum():.2f} GB"
)

Total drive-days: 74,657,725
Total failure rows: 3,128
Total Parquet size: 0.29 GB


In [11]:
for period in ARCHIVES:

    file_path = (
        COMPACT_DIR
        / f"{period}_candidate_drives.parquet"
    )

    models = (
        pd.read_parquet(
            file_path,
            columns=["model"]
        )["model"]
        .value_counts()
    )


    print(
        "\n============================================"
    )

    print(period)

    print(
        "============================================"
    )

    print(models)


2025_Q2
model
TOSHIBA MG08ACA16TA     3650278
TOSHIBA MG07ACA14TA     3400212
ST16000NM001G           3078040
WDC WUH721816ALE6L4     2392488
ST12000NM0008           1718302
ST8000NM0055            1215170
HGST HUH721212ALE604    1207348
HGST HUH721212ALN604     903548
TOSHIBA MG08ACA16TEY     466977
ST24000NM002H            432651
Name: count, dtype: int64

2025_Q3
model
TOSHIBA MG08ACA16TA     3686376
TOSHIBA MG07ACA14TA     3440051
ST16000NM001G           3125133
WDC WUH721816ALE6L4     2425374
ST12000NM0008           1728706
HGST HUH721212ALE604    1227006
ST8000NM0055            1224440
HGST HUH721212ALN604     912361
ST24000NM002H            601539
TOSHIBA MG08ACA16TEY     462943
Name: count, dtype: int64

2025_Q4
model
TOSHIBA MG08ACA16TA     3670933
TOSHIBA MG07ACA14TA     3428534
ST16000NM001G           3149031
WDC WUH721816ALE6L4     2421840
ST12000NM0008           1715709
HGST HUH721212ALE604    1221499
ST8000NM0055            1217828
HGST HUH721212ALN604     900963
ST24000

In [12]:
# ============================================================
# Build failure-event registry across all four quarters
# ============================================================

failure_parts = []


for period in ARCHIVES:

    file_path = (
        COMPACT_DIR
        / f"{period}_candidate_drives.parquet"
    )


    quarter = pd.read_parquet(
        file_path,
        columns=[
            "date",
            "serial_number",
            "model",
            "failure",
        ]
    )


    failed = quarter[
        quarter["failure"].eq(1)
    ].copy()


    if not failed.empty:

        failure_parts.append(
            failed
        )


    print(
        f"{period}: "
        f"{len(failed):,} failure rows"
    )


    del quarter


failure_events = pd.concat(
    failure_parts,
    ignore_index=True
)


failure_events[
    "date"
] = pd.to_datetime(
    failure_events["date"]
)


print(
    "\n============================================"
)
print(
    "FAILURE EVENT REGISTRY"
)
print(
    "============================================"
)

print(
    "Failure rows:",
    f"{len(failure_events):,}"
)

print(
    "Unique failed drives:",
    f"{failure_events['serial_number'].nunique():,}"
)

print(
    "Earliest failure:",
    failure_events["date"].min()
)

print(
    "Latest failure:",
    failure_events["date"].max()
)

2025_Q2: 773 failure rows
2025_Q3: 942 failure rows
2025_Q4: 668 failure rows
2026_Q1: 745 failure rows

FAILURE EVENT REGISTRY
Failure rows: 3,128
Unique failed drives: 3,119
Earliest failure: 2025-04-01 00:00:00
Latest failure: 2026-03-31 00:00:00


In [13]:
failure_counts = (
    failure_events
    .groupby(
        "serial_number"
    )
    .size()
    .sort_values(
        ascending=False
    )
)


print(
    "\n============================================"
)
print(
    "FAILURE RECORD CONSISTENCY"
)
print(
    "============================================"
)

print(
    "Drives with exactly one failure record:",
    int(
        (failure_counts == 1).sum()
    )
)

print(
    "Drives with multiple failure records:",
    int(
        (failure_counts > 1).sum()
    )
)


if (
    failure_counts > 1
).any():

    print(
        "\nRepeated failure records:"
    )

    display(
        failure_counts[
            failure_counts > 1
        ].head(20)
    )


FAILURE RECORD CONSISTENCY
Drives with exactly one failure record: 3110
Drives with multiple failure records: 9

Repeated failure records:


serial_number
22D0A1BWFV8G    2
7060A00UFV8G    2
22C0A018FV8G    2
41D0A096FV8G    2
AAG6J1TH        2
AAG6NL4H        2
ZA1478Y4        2
ZA1819G8        2
ZL005DRH        2
dtype: int64

In [14]:
drive_failure_dates = (
    failure_events
    .groupby(
        "serial_number"
    )
    .agg(
        failure_date=(
            "date",
            "min"
        ),

        drive_model=(
            "model",
            "first"
        ),
    )
    .reset_index()
)


print(
    "\nFailure-date registry:"
)

print(
    "Failed drives:",
    f"{len(drive_failure_dates):,}"
)


display(
    drive_failure_dates.head()
)


Failure-date registry:
Failed drives: 3,119


,serial_number,failure_date,drive_model
0,1060A00RF97G,2025-09-18,TOSHIBA MG07ACA14TA
1,1060A02MF97G,2025-08-29,TOSHIBA MG07ACA14TA
2,1070A03TF97G,2025-10-02,TOSHIBA MG07ACA14TA
3,1070A05AF97G,2025-09-16,TOSHIBA MG07ACA14TA
4,1070A0CGF97G,2025-07-25,TOSHIBA MG07ACA14TA


In [15]:
FAILURE_REGISTRY_FILE = (
    PROCESSED_DIR
    / "maintenance_v2_failure_registry.parquet"
)


drive_failure_dates.to_parquet(
    FAILURE_REGISTRY_FILE,
    index=False
)


print(
    "Saved:",
    FAILURE_REGISTRY_FILE
)

Saved: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_failure_registry.parquet


In [16]:
# ============================================================
# Maintenance v2 labeling configuration
# ============================================================

DATASET_END_DATE = pd.Timestamp(
    "2026-03-31"
)

FAILURE_HORIZON_DAYS = 7

LAST_FULL_HORIZON_DATE = (
    DATASET_END_DATE
    - pd.Timedelta(
        days=FAILURE_HORIZON_DAYS
    )
)


print(
    "\n============================================"
)
print(
    "7-DAY LABEL CONFIGURATION"
)
print(
    "============================================"
)

print(
    "Dataset end:",
    DATASET_END_DATE
)

print(
    "Prediction horizon:",
    FAILURE_HORIZON_DAYS,
    "days"
)

print(
    "Last date with complete "
    "7-day future horizon:",
    LAST_FULL_HORIZON_DATE
)


7-DAY LABEL CONFIGURATION
Dataset end: 2026-03-31 00:00:00
Prediction horizon: 7 days
Last date with complete 7-day future horizon: 2026-03-24 00:00:00


In [17]:
# ============================================================
# Build 7-day predictive-maintenance labels
# ============================================================

LABELED_DIR = (
    PROCESSED_DIR
    / "maintenance_v2_labeled"
)

LABELED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


label_summary = []


for period in ARCHIVES:

    print(
        "\n============================================"
    )
    print(
        f"LABELING {period}"
    )
    print(
        "============================================"
    )


    input_file = (
        COMPACT_DIR
        / f"{period}_candidate_drives.parquet"
    )

    output_file = (
        LABELED_DIR
        / f"{period}_labeled.parquet"
    )


    quarter = pd.read_parquet(
        input_file
    )


    quarter["date"] = pd.to_datetime(
        quarter["date"]
    )


    # ========================================================
    # Attach explicit failure date
    # ========================================================

    quarter = quarter.merge(
        drive_failure_dates[
            [
                "serial_number",
                "failure_date"
            ]
        ],
        on="serial_number",
        how="left",
        validate="many_to_one"
    )


    # ========================================================
    # Days until first recorded failure
    # ========================================================

    quarter[
        "days_until_failure"
    ] = (
        quarter[
            "failure_date"
        ]
        -
        quarter[
            "date"
        ]
    ).dt.days


    # ========================================================
    # Label state
    # ========================================================
    #
    # NORMAL
    #     Sufficient future observation and no failure
    #     within the next seven days.
    #
    # AT_RISK
    #     Explicit failure occurs 1-7 days later.
    #
    # FAILURE_DAY
    #     Observation is the explicit failure day.
    #     Excluded from predictive training/evaluation.
    #
    # POST_FAILURE
    #     Record occurs after the first explicit failure.
    #     Excluded.
    #
    # CENSORED
    #     Insufficient future observation at dataset end.
    # ========================================================

    quarter[
        "label_state"
    ] = "NORMAL"


    # --------------------------------------------------------
    # Dataset-end censoring
    # --------------------------------------------------------

    quarter.loc[
        quarter["date"]
        > LAST_FULL_HORIZON_DATE,
        "label_state"
    ] = "CENSORED"


    # --------------------------------------------------------
    # 1-7 days before explicit failure
    # --------------------------------------------------------

    at_risk_mask = (
        quarter[
            "days_until_failure"
        ].between(
            1,
            FAILURE_HORIZON_DAYS
        )
    )


    quarter.loc[
        at_risk_mask,
        "label_state"
    ] = "AT_RISK"


    # --------------------------------------------------------
    # Explicit failure day
    # --------------------------------------------------------

    failure_day_mask = (
        quarter[
            "days_until_failure"
        ].eq(0)
    )


    quarter.loc[
        failure_day_mask,
        "label_state"
    ] = "FAILURE_DAY"


    # --------------------------------------------------------
    # Records after first failure
    # --------------------------------------------------------

    post_failure_mask = (
        quarter[
            "days_until_failure"
        ].lt(0)
    )


    quarter.loc[
        post_failure_mask,
        "label_state"
    ] = "POST_FAILURE"


    # ========================================================
    # Binary target
    # ========================================================

    quarter[
        "fail_within_7_days"
    ] = pd.NA


    quarter.loc[
        quarter[
            "label_state"
        ].eq("NORMAL"),
        "fail_within_7_days"
    ] = 0


    quarter.loc[
        quarter[
            "label_state"
        ].eq("AT_RISK"),
        "fail_within_7_days"
    ] = 1


    quarter[
        "fail_within_7_days"
    ] = quarter[
        "fail_within_7_days"
    ].astype(
        "Int8"
    )


    # ========================================================
    # Summary
    # ========================================================

    counts = (
        quarter[
            "label_state"
        ]
        .value_counts()
    )


    label_summary.append(
        {
            "period":
                period,

            "rows":
                len(quarter),

            "normal":
                int(
                    counts.get(
                        "NORMAL",
                        0
                    )
                ),

            "at_risk":
                int(
                    counts.get(
                        "AT_RISK",
                        0
                    )
                ),

            "failure_day":
                int(
                    counts.get(
                        "FAILURE_DAY",
                        0
                    )
                ),

            "post_failure":
                int(
                    counts.get(
                        "POST_FAILURE",
                        0
                    )
                ),

            "censored":
                int(
                    counts.get(
                        "CENSORED",
                        0
                    )
                ),
        }
    )


    print(
        counts
    )


    # ========================================================
    # Save
    # ========================================================

    quarter.to_parquet(
        output_file,
        index=False
    )


    print(
        "Saved:",
        output_file
    )


    del quarter


LABELING 2025_Q2
label_state
NORMAL         18459369
AT_RISK            4872
FAILURE_DAY         773
Name: count, dtype: int64
Saved: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_labeled\2025_Q2_labeled.parquet

LABELING 2025_Q3
label_state
NORMAL          18827141
AT_RISK             5804
FAILURE_DAY          942
POST_FAILURE          42
Name: count, dtype: int64
Saved: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_labeled\2025_Q3_labeled.parquet

LABELING 2025_Q4
label_state
NORMAL          18847719
AT_RISK             3676
FAILURE_DAY          659
POST_FAILURE         257
Name: count, dtype: int64
Saved: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_labeled\2025_Q4_labeled.parquet

LABELING 2026_Q1
label_state
NORMAL          17059762
CENSORED         1441409
AT_RISK             4285
FAILURE_DAY          745
POST_FA

In [18]:
label_summary = pd.DataFrame(
    label_summary
)


print(
    "\n============================================"
)
print(
    "7-DAY LABEL SUMMARY"
)
print(
    "============================================"
)


display(
    label_summary
)


7-DAY LABEL SUMMARY


,period,rows,normal,at_risk,failure_day,post_failure,censored
0,2025_Q2,18465014,18459369,4872,773,0,0
1,2025_Q3,18833929,18827141,5804,942,42,0
2,2025_Q4,18852311,18847719,3676,659,257,0
3,2026_Q1,18506471,17059762,4285,745,270,1441409


In [19]:
print(
    "Total AT_RISK drive-days:",
    f"{label_summary['at_risk'].sum():,}"
)

print(
    "Total failure-day rows:",
    f"{label_summary['failure_day'].sum():,}"
)

print(
    "Total censored rows:",
    f"{label_summary['censored'].sum():,}"
)

Total AT_RISK drive-days: 18,637
Total failure-day rows: 3,119
Total censored rows: 1,441,409


In [20]:
# ============================================================
# Validate 7-day target construction
# ============================================================

for period in ARCHIVES:

    file_path = (
        LABELED_DIR
        / f"{period}_labeled.parquet"
    )


    check = pd.read_parquet(
        file_path,
        columns=[
            "label_state",
            "days_until_failure",
            "fail_within_7_days",
        ]
    )


    invalid_positive = check[
        check["label_state"].eq(
            "AT_RISK"
        )
        &
        ~check[
            "days_until_failure"
        ].between(
            1,
            FAILURE_HORIZON_DAYS
        )
    ]


    if len(invalid_positive) > 0:

        raise AssertionError(
            f"{period}: invalid "
            f"AT_RISK labels found."
        )


    invalid_failure_day = check[
        check["label_state"].eq(
            "FAILURE_DAY"
        )
        &
        ~check[
            "days_until_failure"
        ].eq(0)
    ]


    if len(invalid_failure_day) > 0:

        raise AssertionError(
            f"{period}: invalid "
            f"FAILURE_DAY labels found."
        )


    print(
        f"PASS: {period} labels validated."
    )


print(
    "\nPASS: Seven-day target construction "
    "validated across all periods."
)

PASS: 2025_Q2 labels validated.
PASS: 2025_Q3 labels validated.
PASS: 2025_Q4 labels validated.
PASS: 2026_Q1 labels validated.

PASS: Seven-day target construction validated across all periods.


In [21]:
# ============================================================
# Training-only SMART signal analysis
# ============================================================

TRAIN_PERIODS = [
    "2025_Q2",
    "2025_Q3",
]

analysis_parts = []


for period in TRAIN_PERIODS:

    file_path = (
        LABELED_DIR
        / f"{period}_labeled.parquet"
    )

    print(
        f"Loading {period}..."
    )

    df = pd.read_parquet(
        file_path,
        columns=(
            [
                "date",
                "serial_number",
                "model",
                "label_state",
                "days_until_failure",
            ]
            + BASE_SMART_FEATURES
        )
    )

    # Predictive rows only
    df = df[
        df["label_state"].isin(
            [
                "NORMAL",
                "AT_RISK",
            ]
        )
    ].copy()

    analysis_parts.append(
        df
    )


train_analysis = pd.concat(
    analysis_parts,
    ignore_index=True
)


del analysis_parts


print(
    "\n============================================"
)
print(
    "TRAINING SMART ANALYSIS DATA"
)
print(
    "============================================"
)

print(
    "Rows:",
    f"{len(train_analysis):,}"
)

print(
    "\nLabel distribution:"
)

print(
    train_analysis[
        "label_state"
    ].value_counts()
)

Loading 2025_Q2...
Loading 2025_Q3...

TRAINING SMART ANALYSIS DATA
Rows: 37,297,186

Label distribution:
label_state
NORMAL     37286510
AT_RISK       10676
Name: count, dtype: int64


In [22]:
# ============================================================
# NORMAL vs AT_RISK SMART comparison
# ============================================================

comparison_rows = []


for feature in BASE_SMART_FEATURES:

    normal = pd.to_numeric(
        train_analysis.loc[
            train_analysis[
                "label_state"
            ].eq("NORMAL"),
            feature
        ],
        errors="coerce"
    )


    at_risk = pd.to_numeric(
        train_analysis.loc[
            train_analysis[
                "label_state"
            ].eq("AT_RISK"),
            feature
        ],
        errors="coerce"
    )


    comparison_rows.append(
        {
            "feature":
                feature,

            "normal_median":
                normal.median(),

            "at_risk_median":
                at_risk.median(),

            "normal_mean":
                normal.mean(),

            "at_risk_mean":
                at_risk.mean(),

            "normal_p95":
                normal.quantile(
                    0.95
                ),

            "at_risk_p95":
                at_risk.quantile(
                    0.95
                ),

            "normal_nonzero_pct":
                (
                    normal.gt(0).mean()
                    * 100
                ),

            "at_risk_nonzero_pct":
                (
                    at_risk.gt(0).mean()
                    * 100
                ),
        }
    )


smart_failure_comparison = (
    pd.DataFrame(
        comparison_rows
    )
)


print(
    "\n============================================"
)
print(
    "NORMAL vs AT_RISK SMART SIGNALS"
)
print(
    "============================================"
)


display(
    smart_failure_comparison
)


NORMAL vs AT_RISK SMART SIGNALS


,feature,normal_median,at_risk_median,normal_mean,at_risk_mean,normal_p95,at_risk_p95,normal_nonzero_pct,at_risk_nonzero_pct
0,smart_5_raw,0.0,1.0,39.802060,1813.153810,0.0,11401.8,4.553011,49.887598
1,smart_9_raw,31680.0,37496.0,32088.265420,36628.523725,68301.0,67016.3,99.998270,99.681529
2,smart_192_raw,9.0,15.0,639.590397,570.094334,2591.0,2510.0,90.566843,95.794305
3,smart_193_raw,1117.0,1333.0,6565.171943,5438.393310,41812.0,32776.9,99.999920,99.690895
4,smart_194_raw,34.0,34.0,33.234068,33.935263,41.0,42.0,99.999920,99.690895
5,smart_198_raw,0.0,0.0,0.777330,406.823922,0.0,566.0,1.329363,33.617460
6,smart_4_raw,10.0,14.0,15.186628,27.271822,28.0,37.0,99.999429,99.690895
7,smart_12_raw,10.0,14.0,12.896451,26.289674,28.0,37.0,99.996736,99.690895


In [23]:
# ============================================================
# SMART behavior by days before failure
# ============================================================

at_risk_only = train_analysis[
    train_analysis[
        "label_state"
    ].eq(
        "AT_RISK"
    )
].copy()


days_before_summary = (
    at_risk_only
    .groupby(
        "days_until_failure"
    )[
        BASE_SMART_FEATURES
    ]
    .median()
    .sort_index(
        ascending=False
    )
)


print(
    "\n============================================"
)
print(
    "MEDIAN SMART VALUES BEFORE FAILURE"
)
print(
    "============================================"
)


display(
    days_before_summary
)


MEDIAN SMART VALUES BEFORE FAILURE


,smart_5_raw,smart_9_raw,smart_192_raw,smart_193_raw,smart_194_raw,smart_198_raw,smart_4_raw,smart_12_raw
days_until_failure,,,,,,,,
7.0,0.0,37609.0,15.0,1339.5,34.0,0.0,14.0,14.0
6.0,0.0,37620.0,15.0,1346.0,34.0,0.0,14.0,14.0
5.0,0.0,37638.0,15.0,1348.0,34.0,0.0,14.0,14.0
4.0,0.0,37475.0,15.0,1330.5,34.0,0.0,14.0,14.0
3.0,0.0,37246.0,15.0,1325.0,34.0,0.0,14.0,14.0
2.0,1.0,37221.0,15.0,1322.0,34.0,0.0,14.0,14.0
1.0,1.0,37122.0,15.0,1326.0,34.0,0.0,14.0,14.0


In [25]:
# ============================================================
# Save SMART signal analysis
# ============================================================

ANALYSIS_DIR = (
    PROCESSED_DIR
    / "analysis"
)

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


SMART_ANALYSIS_FILE = (
    ANALYSIS_DIR
    / "normal_vs_at_risk_smart_signals.csv"
)


smart_failure_comparison.to_csv(
    SMART_ANALYSIS_FILE,
    index=False
)


print(
    "PASS: SMART signal analysis saved."
)

print(
    "Saved:",
    SMART_ANALYSIS_FILE
)

PASS: SMART signal analysis saved.
Saved: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\analysis\normal_vs_at_risk_smart_signals.csv


### Preliminary SMART Signal Findings

Training-period analysis (2025 Q2-Q3) shows substantial differences
between normal drive-days and observations occurring 1-7 days before
an explicitly recorded failure.

The strongest preliminary signals are SMART 5 (reallocated sectors)
and SMART 198 (offline uncorrectable sectors).

- SMART 5 was non-zero in approximately 4.55% of normal observations
  versus 49.89% of AT_RISK observations.
- SMART 198 was non-zero in approximately 1.33% of normal observations
  versus 33.62% of AT_RISK observations.
- Median SMART 194 temperature was 34 in both groups, indicating that
  temperature alone does not separate the classes at the median.
- Additional SMART attributes show differences in age, cycling, and
  operational-history distributions.

These observations are exploratory and are derived only from the
training periods. Validation and 2026 Q1 test data are not used for
feature selection.

In [26]:
# ============================================================
# Maintenance v2 temporal feature configuration
# ============================================================

TEMPORAL_SMART_FEATURES = [
    "smart_5_raw",
    "smart_9_raw",
    "smart_192_raw",
    "smart_193_raw",
    "smart_194_raw",
    "smart_198_raw",
    "smart_4_raw",
    "smart_12_raw",
]


FEATURE_DIR = (
    PROCESSED_DIR
    / "maintenance_v2_features"
)

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "\n============================================"
)
print(
    "MAINTENANCE V2 FEATURE CONFIGURATION"
)
print(
    "============================================"
)

print(
    "Base SMART features:",
    len(TEMPORAL_SMART_FEATURES)
)

print(
    "Feature output directory:",
    FEATURE_DIR
)


MAINTENANCE V2 FEATURE CONFIGURATION
Base SMART features: 8
Feature output directory: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_features


In [27]:
# ============================================================
# Build temporal SMART features
# ============================================================

def build_temporal_features(df):

    df = df.copy()

    df["date"] = pd.to_datetime(
        df["date"]
    )

    df = (
        df
        .sort_values(
            [
                "serial_number",
                "date"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Ensure SMART values are numeric
    # --------------------------------------------------------

    for feature in TEMPORAL_SMART_FEATURES:

        df[feature] = pd.to_numeric(
            df[feature],
            errors="coerce"
        )


    grouped = df.groupby(
        "serial_number",
        sort=False
    )


    # ========================================================
    # General temporal features
    # ========================================================

    for feature in TEMPORAL_SMART_FEATURES:

        # Previous observation
        previous = grouped[
            feature
        ].shift(1)


        df[
            f"{feature}_delta_1"
        ] = (
            df[feature]
            - previous
        )


        # Seven observations earlier
        previous_7 = grouped[
            feature
        ].shift(7)


        df[
            f"{feature}_delta_7"
        ] = (
            df[feature]
            - previous_7
        )


    # ========================================================
    # Health-specific binary indicators
    # ========================================================

    df[
        "smart_5_nonzero"
    ] = (
        df["smart_5_raw"]
        .fillna(0)
        .gt(0)
        .astype("int8")
    )


    df[
        "smart_198_nonzero"
    ] = (
        df["smart_198_raw"]
        .fillna(0)
        .gt(0)
        .astype("int8")
    )


    df[
        "smart_5_increased"
    ] = (
        df["smart_5_raw_delta_1"]
        .fillna(0)
        .gt(0)
        .astype("int8")
    )


    df[
        "smart_198_increased"
    ] = (
        df["smart_198_raw_delta_1"]
        .fillna(0)
        .gt(0)
        .astype("int8")
    )


    # ========================================================
    # Temperature features
    # ========================================================

    temperature_grouped = (
        df.groupby(
            "serial_number",
            sort=False
        )[
            "smart_194_raw"
        ]
    )


    df[
        "temperature_7obs_mean"
    ] = (
        temperature_grouped
        .rolling(
            window=7,
            min_periods=2
        )
        .mean()
        .reset_index(
            level=0,
            drop=True
        )
    )


    df[
        "temperature_7obs_max"
    ] = (
        temperature_grouped
        .rolling(
            window=7,
            min_periods=2
        )
        .max()
        .reset_index(
            level=0,
            drop=True
        )
    )


    df[
        "temperature_7obs_min"
    ] = (
        temperature_grouped
        .rolling(
            window=7,
            min_periods=2
        )
        .min()
        .reset_index(
            level=0,
            drop=True
        )
    )


    df[
        "temperature_7obs_range"
    ] = (
        df["temperature_7obs_max"]
        - df["temperature_7obs_min"]
    )


    df[
        "temperature_vs_7obs_mean"
    ] = (
        df["smart_194_raw"]
        - df["temperature_7obs_mean"]
    )


    return df

In [28]:
# ============================================================
# Test feature engineering on 2025 Q2
# ============================================================

TEST_PERIOD = "2025_Q2"


input_file = (
    LABELED_DIR
    / f"{TEST_PERIOD}_labeled.parquet"
)


print(
    "Loading:",
    input_file
)


q2 = pd.read_parquet(
    input_file
)


print(
    "Rows:",
    f"{len(q2):,}"
)

print(
    "Columns before:",
    len(q2.columns)
)


q2_features = build_temporal_features(
    q2
)


print(
    "Columns after:",
    len(q2_features.columns)
)

Loading: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_labeled\2025_Q2_labeled.parquet
Rows: 18,465,014
Columns before: 17
Columns after: 42


In [29]:
# ============================================================
# Feature validation
# ============================================================

ENGINEERED_FEATURES = [
    "smart_5_raw_delta_1",
    "smart_5_raw_delta_7",
    "smart_198_raw_delta_1",
    "smart_198_raw_delta_7",
    "smart_5_nonzero",
    "smart_198_nonzero",
    "smart_5_increased",
    "smart_198_increased",
    "temperature_7obs_mean",
    "temperature_7obs_max",
    "temperature_7obs_min",
    "temperature_7obs_range",
    "temperature_vs_7obs_mean",
]


print(
    "\n============================================"
)
print(
    "ENGINEERED FEATURE VALIDATION"
)
print(
    "============================================"
)


for feature in ENGINEERED_FEATURES:

    if feature not in q2_features.columns:

        raise AssertionError(
            f"Missing feature: {feature}"
        )


    print(
        f"{feature:<32}",
        "missing =",
        f"{q2_features[feature].isna().mean() * 100:.2f}%"
    )


print(
    "\nPASS: Engineered features created."
)


ENGINEERED FEATURE VALIDATION
smart_5_raw_delta_1              missing = 1.11%
smart_5_raw_delta_7              missing = 7.78%
smart_198_raw_delta_1            missing = 1.11%
smart_198_raw_delta_7            missing = 7.78%
smart_5_nonzero                  missing = 0.00%
smart_198_nonzero                missing = 0.00%
smart_5_increased                missing = 0.00%
smart_198_increased              missing = 0.00%
temperature_7obs_mean            missing = 1.11%
temperature_7obs_max             missing = 1.11%
temperature_7obs_min             missing = 1.11%
temperature_7obs_range           missing = 1.11%
temperature_vs_7obs_mean         missing = 1.11%

PASS: Engineered features created.


In [30]:
example_serial = (
    q2_features[
        "serial_number"
    ]
    .value_counts()
    .index[0]
)


example = q2_features[
    q2_features[
        "serial_number"
    ].eq(
        example_serial
    )
][
    [
        "date",
        "serial_number",
        "model",
        "smart_5_raw",
        "smart_5_raw_delta_1",
        "smart_5_raw_delta_7",
        "smart_194_raw",
        "temperature_7obs_mean",
        "temperature_7obs_range",
    ]
].head(15)


display(
    example
)

,date,serial_number,model,smart_5_raw,smart_5_raw_delta_1,smart_5_raw_delta_7,smart_194_raw,temperature_7obs_mean,temperature_7obs_range
180,2025-04-01,1030A00DF97G,TOSHIBA MG07ACA14TA,0.0,NaN,NaN,27.0,NaN,NaN
181,2025-04-02,1030A00DF97G,TOSHIBA MG07ACA14TA,0.0,0.0,NaN,27.0,27.000000,0.0
182,2025-04-03,1030A00DF97G,TOSHIBA MG07ACA14TA,0.0,0.0,NaN,27.0,27.000000,0.0
183,2025-04-04,1030A00DF97G,TOSHIBA MG07ACA14TA,0.0,0.0,NaN,27.0,27.000000,0.0
184,2025-04-05,1030A00DF97G,TOSHIBA MG07ACA14TA,0.0,0.0,NaN,27.0,27.000000,0.0
185,2025-04-06,1030A00DF97G,TOSHIBA MG07ACA14TA,0.0,0.0,NaN,27.0,27.000000,0.0
186,2025-04-07,1030A00DF97G,TOSHIBA MG07ACA14TA,0.0,0.0,NaN,27.0,27.000000,0.0
187,2025-04-08,1030A00DF97G,TOSHIBA MG07ACA14TA,0.0,0.0,0.0,28.0,27.142857,1.0
188,2025-04-09,1030A00DF97G,TOSHIBA MG07ACA14TA,0.0,0.0,0.0,27.0,27.142857,1.0
189,2025-04-10,1030A00DF97G,TOSHIBA MG07ACA14TA,0.0,0.0,0.0,27.0,27.142857,1.0


In [31]:
# ============================================================
# Previous-quarter history helper
# ============================================================

HISTORY_DAYS = 7


def get_history_tail(
    previous_df,
    current_start_date,
    history_days=HISTORY_DAYS
):

    history_start = (
        current_start_date
        - pd.Timedelta(
            days=history_days
        )
    )

    history = previous_df[
        (
            previous_df["date"]
            >= history_start
        )
        &
        (
            previous_df["date"]
            < current_start_date
        )
    ].copy()

    return history

In [32]:
# ============================================================
# Build final Maintenance v2 temporal feature datasets
# ============================================================

PERIOD_ORDER = [
    "2025_Q2",
    "2025_Q3",
    "2025_Q4",
    "2026_Q1",
]


previous_labeled = None


feature_build_summary = []


for period in PERIOD_ORDER:

    print(
        "\n============================================"
    )
    print(
        f"BUILDING FEATURES: {period}"
    )
    print(
        "============================================"
    )


    input_file = (
        LABELED_DIR
        / f"{period}_labeled.parquet"
    )


    current = pd.read_parquet(
        input_file
    )


    current["date"] = pd.to_datetime(
        current["date"]
    )


    current_start = (
        current["date"].min()
    )


    current_end = (
        current["date"].max()
    )


    print(
        "Current period:",
        current_start,
        "→",
        current_end
    )


    # ========================================================
    # Add history from previous quarter
    # ========================================================

    if previous_labeled is not None:

        history = get_history_tail(
            previous_labeled,
            current_start
        )


        print(
            "History rows carried:",
            f"{len(history):,}"
        )


        combined = pd.concat(
            [
                history,
                current
            ],
            ignore_index=True
        )

    else:

        history = None

        combined = current.copy()

        print(
            "History rows carried: 0 "
            "(first available period)"
        )


    # ========================================================
    # Build temporal features
    # ========================================================

    featured = build_temporal_features(
        combined
    )


    # ========================================================
    # Keep only current-period rows
    # ========================================================

    current_features = featured[
        (
            featured["date"]
            >= current_start
        )
        &
        (
            featured["date"]
            <= current_end
        )
    ].copy()


    # Safety check
    if len(current_features) != len(current):

        raise AssertionError(
            f"{period}: row count changed "
            f"during feature engineering."
        )


    # ========================================================
    # Save
    # ========================================================

    output_file = (
        FEATURE_DIR
        / f"{period}_features.parquet"
    )


    current_features.to_parquet(
        output_file,
        index=False
    )


    feature_build_summary.append(
        {
            "period":
                period,

            "rows":
                len(current_features),

            "start_date":
                current_start,

            "end_date":
                current_end,

            "history_rows":
                (
                    0
                    if history is None
                    else len(history)
                ),

            "columns":
                len(current_features.columns),
        }
    )


    print(
        "Rows saved:",
        f"{len(current_features):,}"
    )

    print(
        "Columns:",
        len(current_features.columns)
    )

    print(
        "Saved:",
        output_file
    )


    # ========================================================
    # Current quarter becomes history source for next quarter
    # ========================================================

    previous_labeled = current[
            current["date"]
            >= (
                current_end
                - pd.Timedelta(days=HISTORY_DAYS)
            )
        ].copy()

    del current
    # Free large temporary frames
    del combined
    del featured
    del current_features

    if history is not None:
        del history


BUILDING FEATURES: 2025_Q2
Current period: 2025-04-01 00:00:00 → 2025-06-30 00:00:00
History rows carried: 0 (first available period)
Rows saved: 18,465,014
Columns: 42
Saved: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_features\2025_Q2_features.parquet

BUILDING FEATURES: 2025_Q3
Current period: 2025-07-01 00:00:00 → 2025-09-30 00:00:00
History rows carried: 1,431,191
Rows saved: 18,833,929
Columns: 42
Saved: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_features\2025_Q3_features.parquet

BUILDING FEATURES: 2025_Q4
Current period: 2025-10-01 00:00:00 → 2025-12-31 00:00:00
History rows carried: 1,435,719
Rows saved: 18,852,311
Columns: 42
Saved: C:\Users\Aljawharah\Documents\DC-Guardian\phase1\predictive_maintenance\data\processed\maintenance_v2_features\2025_Q4_features.parquet

BUILDING FEATURES: 2026_Q1
Current period: 2026-01-01 00:00:00 → 2026-03-31 00:00:00
History r

In [33]:
feature_build_summary = pd.DataFrame(
    feature_build_summary
)


print(
    "\n============================================"
)
print(
    "MAINTENANCE V2 FEATURE BUILD SUMMARY"
)
print(
    "============================================"
)


display(
    feature_build_summary
)


MAINTENANCE V2 FEATURE BUILD SUMMARY


,period,rows,start_date,end_date,history_rows,columns
0,2025_Q2,18465014,2025-04-01,2025-06-30,0,42
1,2025_Q3,18833929,2025-07-01,2025-09-30,1431191,42
2,2025_Q4,18852311,2025-10-01,2025-12-31,1435719,42
3,2026_Q1,18506471,2026-01-01,2026-03-31,1436311,42


In [34]:
# ============================================================
# Maintenance v2 feature / split integrity validation
# ============================================================

SPLIT_CONFIG = {
    "2025_Q2": "TRAIN",
    "2025_Q3": "TRAIN",
    "2025_Q4": "VALIDATION",
    "2026_Q1": "TEST",
}


EXPECTED_PERIODS = {
    "2025_Q2": (
        pd.Timestamp("2025-04-01"),
        pd.Timestamp("2025-06-30"),
    ),

    "2025_Q3": (
        pd.Timestamp("2025-07-01"),
        pd.Timestamp("2025-09-30"),
    ),

    "2025_Q4": (
        pd.Timestamp("2025-10-01"),
        pd.Timestamp("2025-12-31"),
    ),

    "2026_Q1": (
        pd.Timestamp("2026-01-01"),
        pd.Timestamp("2026-03-31"),
    ),
}


integrity_rows = []


for period, split_name in SPLIT_CONFIG.items():

    print(
        "\n============================================"
    )
    print(
        f"VALIDATING {period} / {split_name}"
    )
    print(
        "============================================"
    )


    file_path = (
        FEATURE_DIR
        / f"{period}_features.parquet"
    )


    # Only read columns needed for validation.
    df = pd.read_parquet(
        file_path,
        columns=[
            "date",
            "serial_number",
            "model",
            "failure",
            "failure_date",
            "days_until_failure",
            "label_state",
            "fail_within_7_days",
        ]
    )


    df["date"] = pd.to_datetime(
        df["date"]
    )


    expected_start, expected_end = (
        EXPECTED_PERIODS[period]
    )


    # ========================================================
    # Date boundaries
    # ========================================================

    actual_start = df["date"].min()
    actual_end = df["date"].max()


    if actual_start != expected_start:

        raise AssertionError(
            f"{period}: unexpected start date "
            f"{actual_start}"
        )


    if actual_end != expected_end:

        raise AssertionError(
            f"{period}: unexpected end date "
            f"{actual_end}"
        )


    # ========================================================
    # Failure-day rows must never be predictive positives
    # ========================================================

    failure_day = df[
        df["label_state"].eq(
            "FAILURE_DAY"
        )
    ]


    if (
        failure_day[
            "fail_within_7_days"
        ]
        .notna()
        .any()
    ):

        raise AssertionError(
            f"{period}: FAILURE_DAY rows "
            f"have predictive labels."
        )


    # ========================================================
    # Censored rows must not have predictive labels
    # ========================================================

    censored = df[
        df["label_state"].eq(
            "CENSORED"
        )
    ]


    if (
        censored[
            "fail_within_7_days"
        ]
        .notna()
        .any()
    ):

        raise AssertionError(
            f"{period}: CENSORED rows "
            f"have predictive labels."
        )


    # ========================================================
    # AT_RISK must mean failure in 1-7 days
    # ========================================================

    at_risk = df[
        df["label_state"].eq(
            "AT_RISK"
        )
    ]


    invalid_at_risk = at_risk[
        ~at_risk[
            "days_until_failure"
        ].between(
            1,
            FAILURE_HORIZON_DAYS
        )
    ]


    if not invalid_at_risk.empty:

        raise AssertionError(
            f"{period}: invalid AT_RISK "
            f"rows found."
        )


    # ========================================================
    # NORMAL cannot contain a known failure <= 7 days away
    # ========================================================

    normal = df[
        df["label_state"].eq(
            "NORMAL"
        )
    ]


    invalid_normal = normal[
        normal[
            "days_until_failure"
        ].between(
            1,
            FAILURE_HORIZON_DAYS
        )
    ]


    if not invalid_normal.empty:

        raise AssertionError(
            f"{period}: NORMAL rows contain "
            f"known <=7-day failures."
        )


    # ========================================================
    # Summary
    # ========================================================

    predictive = df[
        df[
            "fail_within_7_days"
        ].notna()
    ]


    positive_rows = int(
        predictive[
            "fail_within_7_days"
        ].sum()
    )


    positive_drives = (
        predictive.loc[
            predictive[
                "fail_within_7_days"
            ].eq(1),
            "serial_number"
        ]
        .nunique()
    )


    integrity_rows.append(
        {
            "period":
                period,

            "split":
                split_name,

            "rows":
                len(df),

            "predictive_rows":
                len(predictive),

            "positive_rows":
                positive_rows,

            "positive_drives":
                positive_drives,

            "failure_day_rows":
                len(failure_day),

            "censored_rows":
                len(censored),
        }
    )


    print(
        "PASS: Date boundaries"
    )

    print(
        "PASS: Failure-day exclusion"
    )

    print(
        "PASS: Censoring exclusion"
    )

    print(
        "PASS: 7-day target integrity"
    )


    del df


integrity_summary = pd.DataFrame(
    integrity_rows
)


print(
    "\n============================================"
)
print(
    "MAINTENANCE V2 SPLIT INTEGRITY SUMMARY"
)
print(
    "============================================"
)


display(
    integrity_summary
)


print(
    "\nPASS: Maintenance v2 temporal "
    "split integrity validated."
)


VALIDATING 2025_Q2 / TRAIN
PASS: Date boundaries
PASS: Failure-day exclusion
PASS: Censoring exclusion
PASS: 7-day target integrity

VALIDATING 2025_Q3 / TRAIN
PASS: Date boundaries
PASS: Failure-day exclusion
PASS: Censoring exclusion
PASS: 7-day target integrity

VALIDATING 2025_Q4 / VALIDATION
PASS: Date boundaries
PASS: Failure-day exclusion
PASS: Censoring exclusion
PASS: 7-day target integrity

VALIDATING 2026_Q1 / TEST
PASS: Date boundaries
PASS: Failure-day exclusion
PASS: Censoring exclusion
PASS: 7-day target integrity

MAINTENANCE V2 SPLIT INTEGRITY SUMMARY


,period,split,rows,predictive_rows,positive_rows,positive_drives,failure_day_rows,censored_rows
0,2025_Q2,TRAIN,18465014,18464241,4872,802,773,0
1,2025_Q3,TRAIN,18833929,18832945,5804,945,942,0
2,2025_Q4,VALIDATION,18852311,18851395,3676,631,659,0
3,2026_Q1,TEST,18506471,17064047,4285,667,745,1441409



PASS: Maintenance v2 temporal split integrity validated.


In [35]:
# ============================================================
# Maintenance v2 modeling configuration
# ============================================================

TRAIN_PERIODS = [
    "2025_Q2",
    "2025_Q3",
]

VALIDATION_PERIOD = "2025_Q4"

TEST_PERIOD = "2026_Q1"

NEGATIVE_TO_POSITIVE_RATIO = 20

RANDOM_STATE = 42


# ============================================================
# Baseline A features
# ============================================================

BASELINE_A_FEATURES = [
    "smart_5_raw",
    "smart_9_raw",
    "smart_192_raw",
    "smart_193_raw",
    "smart_194_raw",
    "smart_198_raw",
    "smart_4_raw",
    "smart_12_raw",
]


print(
    "Negative : positive training ratio:",
    f"{NEGATIVE_TO_POSITIVE_RATIO}:1"
)

print(
    "Baseline A features:",
    len(BASELINE_A_FEATURES)
)

Negative : positive training ratio: 20:1
Baseline A features: 8


In [36]:
# ============================================================
# Build sampled TRAIN dataset
# ============================================================

train_positive_parts = []
train_negative_parts = []


for period in TRAIN_PERIODS:

    print(
        "\nLoading:",
        period
    )

    file_path = (
        FEATURE_DIR
        / f"{period}_features.parquet"
    )

    columns = (
        [
            "date",
            "serial_number",
            "model",
            "fail_within_7_days",
        ]
        + BASELINE_A_FEATURES
    )


    df = pd.read_parquet(
        file_path,
        columns=columns
    )


    # --------------------------------------------------------
    # Predictive rows only
    # --------------------------------------------------------

    df = df[
        df[
            "fail_within_7_days"
        ].notna()
    ].copy()


    positive = df[
        df[
            "fail_within_7_days"
        ].eq(1)
    ].copy()


    negative = df[
        df[
            "fail_within_7_days"
        ].eq(0)
    ].copy()


    train_positive_parts.append(
        positive
    )


    # --------------------------------------------------------
    # Temporary storage of negatives
    #
    # We will sample after knowing the total positive count.
    # --------------------------------------------------------

    train_negative_parts.append(
        negative
    )


    print(
        "  Positive:",
        f"{len(positive):,}"
    )

    print(
        "  Negative:",
        f"{len(negative):,}"
    )


    del df


Loading: 2025_Q2
  Positive: 4,872
  Negative: 18,459,369

Loading: 2025_Q3
  Positive: 5,804
  Negative: 18,827,141


In [37]:
train_positive = pd.concat(
    train_positive_parts,
    ignore_index=True
)


total_positive = len(
    train_positive
)


target_negative_count = (
    total_positive
    * NEGATIVE_TO_POSITIVE_RATIO
)


print(
    "\n============================================"
)
print(
    "TRAINING SAMPLE TARGET"
)
print(
    "============================================"
)

print(
    "Positive rows:",
    f"{total_positive:,}"
)

print(
    "Target negative rows:",
    f"{target_negative_count:,}"
)


TRAINING SAMPLE TARGET
Positive rows: 10,676
Target negative rows: 213,520


In [38]:
# ============================================================
# Sample normal training observations
# ============================================================

all_negative_count = sum(
    len(part)
    for part in train_negative_parts
)


sampled_negative_parts = []


for negative in train_negative_parts:

    proportion = (
        len(negative)
        / all_negative_count
    )


    sample_size = int(
        round(
            target_negative_count
            * proportion
        )
    )


    sampled = negative.sample(
        n=min(
            sample_size,
            len(negative)
        ),
        random_state=RANDOM_STATE
    )


    sampled_negative_parts.append(
        sampled
    )


train_negative = pd.concat(
    sampled_negative_parts,
    ignore_index=True
)


train_model_df = pd.concat(
    [
        train_positive,
        train_negative,
    ],
    ignore_index=True
)


# Shuffle training observations
train_model_df = (
    train_model_df
    .sample(
        frac=1.0,
        random_state=RANDOM_STATE
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n============================================"
)
print(
    "SAMPLED TRAINING DATA"
)
print(
    "============================================"
)

print(
    "Rows:",
    f"{len(train_model_df):,}"
)

print(
    "\nClass distribution:"
)

print(
    train_model_df[
        "fail_within_7_days"
    ].value_counts()
)


SAMPLED TRAINING DATA
Rows: 224,196

Class distribution:
fail_within_7_days
0    213520
1     10676
Name: count, dtype: Int64


In [39]:
del train_negative_parts
del sampled_negative_parts
del train_positive_parts

In [40]:
# ============================================================
# Training sample coverage
# ============================================================

sample_model_summary = (
    train_model_df
    .groupby(
        [
            "model",
            "fail_within_7_days",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


print(
    "\n============================================"
)
print(
    "TRAINING SAMPLE BY DRIVE MODEL"
)
print(
    "============================================"
)


display(
    sample_model_summary
)


TRAINING SAMPLE BY DRIVE MODEL


fail_within_7_days,0,1
model,,
HGST HUH721212ALE604,13844,1025
HGST HUH721212ALN604,10528,988
ST12000NM0008,19749,1668
ST16000NM001G,35491,665
ST24000NM002H,5997,517
ST8000NM0055,13857,642
TOSHIBA MG07ACA14TA,39180,1357
TOSHIBA MG08ACA16TA,41895,1368
TOSHIBA MG08ACA16TEY,5410,1781


In [41]:
# ============================================================
# Random Forest Baseline A
#
# Current SMART values only
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


X_train_A = train_model_df[
    BASELINE_A_FEATURES
].copy()


y_train_A = (
    train_model_df[
        "fail_within_7_days"
    ]
    .astype(int)
)


baseline_a_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=300,

                max_depth=None,

                min_samples_leaf=2,

                max_features="sqrt",

                random_state=RANDOM_STATE,

                n_jobs=-1,
            )
        ),
    ]
)


print(
    "\n============================================"
)
print(
    "TRAINING RANDOM FOREST BASELINE A"
)
print(
    "============================================"
)

print(
    "Training rows:",
    f"{len(X_train_A):,}"
)

print(
    "Features:",
    len(BASELINE_A_FEATURES)
)


baseline_a_pipeline.fit(
    X_train_A,
    y_train_A
)


print(
    "\nPASS: Baseline A trained."
)


TRAINING RANDOM FOREST BASELINE A
Training rows: 224,196
Features: 8

PASS: Baseline A trained.


In [42]:
# ============================================================
# Load natural-prevalence validation data
# ============================================================

validation_file = (
    FEATURE_DIR
    / "2025_Q4_features.parquet"
)


validation_df = pd.read_parquet(
    validation_file,
    columns=(
        [
            "serial_number",
            "model",
            "fail_within_7_days",
        ]
        + BASELINE_A_FEATURES
    )
)


validation_df = validation_df[
    validation_df[
        "fail_within_7_days"
    ].notna()
].copy()


X_validation_A = validation_df[
    BASELINE_A_FEATURES
]


y_validation = (
    validation_df[
        "fail_within_7_days"
    ]
    .astype(int)
)


print(
    "\n============================================"
)
print(
    "VALIDATION DATA"
)
print(
    "============================================"
)

print(
    "Rows:",
    f"{len(validation_df):,}"
)

print(
    "Positive rows:",
    f"{y_validation.sum():,}"
)

print(
    "Positive drives:",
    validation_df.loc[
        y_validation.eq(1),
        "serial_number"
    ].nunique()
)


VALIDATION DATA
Rows: 18,851,395
Positive rows: 3,676
Positive drives: 631


In [43]:
# ============================================================
# Baseline A validation probabilities
# ============================================================

validation_probability_A = (
    baseline_a_pipeline
    .predict_proba(
        X_validation_A
    )[:, 1]
)


print(
    "PASS: Validation probabilities generated."
)

print(
    "Minimum probability:",
    validation_probability_A.min()
)

print(
    "Maximum probability:",
    validation_probability_A.max()
)

print(
    "Mean probability:",
    validation_probability_A.mean()
)

PASS: Validation probabilities generated.
Minimum probability: 0.0
Maximum probability: 0.9793639406931165
Mean probability: 0.02147773866874902


In [44]:
# ============================================================
# Threshold-independent validation metrics
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


pr_auc_A = average_precision_score(
    y_validation,
    validation_probability_A
)


roc_auc_A = roc_auc_score(
    y_validation,
    validation_probability_A
)


validation_prevalence = (
    y_validation.mean()
)


print(
    "\n============================================"
)
print(
    "BASELINE A - Q4 VALIDATION"
)
print(
    "============================================"
)

print(
    "Validation prevalence:",
    f"{validation_prevalence:.6%}"
)

print(
    "PR-AUC:",
    f"{pr_auc_A:.6f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_A:.6f}"
)


BASELINE A - Q4 VALIDATION
Validation prevalence: 0.019500%
PR-AUC: 0.017609
ROC-AUC: 0.868802


In [45]:
# ============================================================
# Threshold-independent validation metrics
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


pr_auc_A = average_precision_score(
    y_validation,
    validation_probability_A
)


roc_auc_A = roc_auc_score(
    y_validation,
    validation_probability_A
)


validation_prevalence = (
    y_validation.mean()
)


print(
    "\n============================================"
)
print(
    "BASELINE A - Q4 VALIDATION"
)
print(
    "============================================"
)

print(
    "Validation prevalence:",
    f"{validation_prevalence:.6%}"
)

print(
    "PR-AUC:",
    f"{pr_auc_A:.6f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_A:.6f}"
)


BASELINE A - Q4 VALIDATION
Validation prevalence: 0.019500%
PR-AUC: 0.017609
ROC-AUC: 0.868802


In [46]:
random_pr_baseline = (
    y_validation.mean()
)

pr_lift_A = (
    pr_auc_A
    / random_pr_baseline
)


print(
    "Random PR baseline:",
    f"{random_pr_baseline:.6f}"
)

print(
    "PR-AUC:",
    f"{pr_auc_A:.6f}"
)

print(
    "PR lift over prevalence:",
    f"{pr_lift_A:.1f}x"
)

Random PR baseline: 0.000195
PR-AUC: 0.017609
PR lift over prevalence: 90.3x


In [47]:
# ============================================================
# Validation threshold sensitivity
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)


THRESHOLDS = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
]


threshold_rows = []


for threshold in THRESHOLDS:

    prediction = (
        validation_probability_A
        >= threshold
    ).astype(int)


    precision = precision_score(
        y_validation,
        prediction,
        zero_division=0
    )


    recall = recall_score(
        y_validation,
        prediction,
        zero_division=0
    )


    f1 = f1_score(
        y_validation,
        prediction,
        zero_division=0
    )


    alerts = int(
        prediction.sum()
    )


    true_positive_drives = (
        validation_df.loc[
            (
                prediction == 1
            )
            &
            (
                y_validation.to_numpy()
                == 1
            ),
            "serial_number"
        ]
        .nunique()
    )


    total_positive_drives = (
        validation_df.loc[
            y_validation.eq(1),
            "serial_number"
        ]
        .nunique()
    )


    drive_recall = (
        true_positive_drives
        / total_positive_drives
    )


    alert_rate = (
        alerts
        / len(
            validation_df
        )
    )


    alerts_per_1000 = (
        alert_rate
        * 1000
    )


    threshold_rows.append(
        {
            "threshold":
                threshold,

            "precision":
                precision,

            "recall":
                recall,

            "f1":
                f1,

            "alerts":
                alerts,

            "alert_rate_pct":
                alert_rate * 100,

            "alerts_per_1000_drive_days":
                alerts_per_1000,

            "detected_failed_drives":
                true_positive_drives,

            "total_failed_drives":
                total_positive_drives,

            "drive_recall":
                drive_recall,
        }
    )


baseline_a_thresholds = pd.DataFrame(
    threshold_rows
)


display(
    baseline_a_thresholds
)

,threshold,precision,recall,f1,alerts,alert_rate_pct,alerts_per_1000_drive_days,detected_failed_drives,total_failed_drives,drive_recall
0,0.05,0.001356,0.720620,0.002708,1953087,10.360438,103.604375,500,631,0.792393
1,0.10,0.002240,0.626496,0.004463,1028344,5.455002,54.550021,446,631,0.706815
2,0.15,0.003147,0.552231,0.006259,645036,3.421688,34.216884,407,631,0.645008
3,0.20,0.004210,0.495375,0.008348,432578,2.294674,22.946737,368,631,0.583201
4,0.25,0.005401,0.440968,0.010672,300114,1.591999,15.919989,333,631,0.527734
5,0.30,0.006748,0.396627,0.013270,216071,1.146180,11.461804,306,631,0.484945
6,0.40,0.010365,0.308760,0.020056,109507,0.580896,5.808960,250,631,0.396197
7,0.50,0.015375,0.224701,0.028781,53722,0.284976,2.849763,191,631,0.302694


In [48]:
# ============================================================
# Baseline B features
#
# Current SMART values
# + explicit health indicators
# ============================================================

BASELINE_B_FEATURES = (
    BASELINE_A_FEATURES
    + [
        "smart_5_nonzero",
        "smart_198_nonzero",
        "smart_5_increased",
        "smart_198_increased",
    ]
)


print(
    "\n============================================"
)
print(
    "BASELINE B FEATURES"
)
print(
    "============================================"
)

print(
    "Features:",
    len(BASELINE_B_FEATURES)
)

for feature in BASELINE_B_FEATURES:
    print(feature)


BASELINE B FEATURES
Features: 12
smart_5_raw
smart_9_raw
smart_192_raw
smart_193_raw
smart_194_raw
smart_198_raw
smart_4_raw
smart_12_raw
smart_5_nonzero
smart_198_nonzero
smart_5_increased
smart_198_increased


In [49]:
# ============================================================
# Preserve sampled training observation keys
# ============================================================

TRAIN_SAMPLE_KEYS = (
    train_model_df[
        [
            "date",
            "serial_number",
        ]
    ]
    .drop_duplicates()
)


print(
    "Sampled observation keys:",
    f"{len(TRAIN_SAMPLE_KEYS):,}"
)

Sampled observation keys: 224,196


In [50]:
# ============================================================
# Build Baseline B training dataset
# ============================================================

baseline_b_train_parts = []


for period in TRAIN_PERIODS:

    file_path = (
        FEATURE_DIR
        / f"{period}_features.parquet"
    )


    df = pd.read_parquet(
        file_path,
        columns=(
            [
                "date",
                "serial_number",
                "fail_within_7_days",
            ]
            + BASELINE_B_FEATURES
        )
    )


    df = df.merge(
        TRAIN_SAMPLE_KEYS,
        on=[
            "date",
            "serial_number",
        ],
        how="inner",
        validate="one_to_one"
    )


    baseline_b_train_parts.append(
        df
    )


train_B = pd.concat(
    baseline_b_train_parts,
    ignore_index=True
)


del baseline_b_train_parts


print(
    "Baseline B training rows:",
    f"{len(train_B):,}"
)

print(
    train_B[
        "fail_within_7_days"
    ].value_counts()
)

Baseline B training rows: 224,196
fail_within_7_days
0    213520
1     10676
Name: count, dtype: Int64


In [51]:
# ============================================================
# Train Random Forest Baseline B
# ============================================================

X_train_B = train_B[
    BASELINE_B_FEATURES
]

y_train_B = (
    train_B[
        "fail_within_7_days"
    ]
    .astype(int)
)


baseline_b_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        ),
    ]
)


baseline_b_pipeline.fit(
    X_train_B,
    y_train_B
)


print(
    "PASS: Baseline B trained."
)

PASS: Baseline B trained.


In [52]:
validation_B = pd.read_parquet(
    validation_file,
    columns=(
        [
            "serial_number",
            "fail_within_7_days",
        ]
        + BASELINE_B_FEATURES
    )
)


validation_B = validation_B[
    validation_B[
        "fail_within_7_days"
    ].notna()
].copy()


X_validation_B = validation_B[
    BASELINE_B_FEATURES
]


validation_probability_B = (
    baseline_b_pipeline
    .predict_proba(
        X_validation_B
    )[:, 1]
)


pr_auc_B = average_precision_score(
    y_validation,
    validation_probability_B
)


roc_auc_B = roc_auc_score(
    y_validation,
    validation_probability_B
)


print(
    "\n============================================"
)
print(
    "BASELINE B - Q4 VALIDATION"
)
print(
    "============================================"
)

print(
    "PR-AUC:",
    f"{pr_auc_B:.6f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_B:.6f}"
)

print(
    "PR-AUC change vs A:",
    f"{pr_auc_B - pr_auc_A:+.6f}"
)


BASELINE B - Q4 VALIDATION
PR-AUC: 0.021600
ROC-AUC: 0.870877
PR-AUC change vs A: +0.003991


In [53]:
# ============================================================
# Temporal RF v2 feature set
# ============================================================

TEMPORAL_RF_FEATURES = (
    BASELINE_B_FEATURES
    + [
        # SMART 5 change
        "smart_5_raw_delta_1",
        "smart_5_raw_delta_7",

        # SMART 9 change
        "smart_9_raw_delta_1",
        "smart_9_raw_delta_7",

        # SMART 192 change
        "smart_192_raw_delta_1",
        "smart_192_raw_delta_7",

        # SMART 193 change
        "smart_193_raw_delta_1",
        "smart_193_raw_delta_7",

        # SMART 194 temperature change
        "smart_194_raw_delta_1",
        "smart_194_raw_delta_7",

        # SMART 198 change
        "smart_198_raw_delta_1",
        "smart_198_raw_delta_7",

        # SMART 4 change
        "smart_4_raw_delta_1",
        "smart_4_raw_delta_7",

        # SMART 12 change
        "smart_12_raw_delta_1",
        "smart_12_raw_delta_7",

        # Temperature history
        "temperature_7obs_mean",
        "temperature_7obs_max",
        "temperature_7obs_min",
        "temperature_7obs_range",
        "temperature_vs_7obs_mean",
    ]
)


print(
    "\n============================================"
)
print(
    "TEMPORAL RF V2 FEATURES"
)
print(
    "============================================"
)

print(
    "Total features:",
    len(TEMPORAL_RF_FEATURES)
)

for feature in TEMPORAL_RF_FEATURES:
    print(feature)


TEMPORAL RF V2 FEATURES
Total features: 33
smart_5_raw
smart_9_raw
smart_192_raw
smart_193_raw
smart_194_raw
smart_198_raw
smart_4_raw
smart_12_raw
smart_5_nonzero
smart_198_nonzero
smart_5_increased
smart_198_increased
smart_5_raw_delta_1
smart_5_raw_delta_7
smart_9_raw_delta_1
smart_9_raw_delta_7
smart_192_raw_delta_1
smart_192_raw_delta_7
smart_193_raw_delta_1
smart_193_raw_delta_7
smart_194_raw_delta_1
smart_194_raw_delta_7
smart_198_raw_delta_1
smart_198_raw_delta_7
smart_4_raw_delta_1
smart_4_raw_delta_7
smart_12_raw_delta_1
smart_12_raw_delta_7
temperature_7obs_mean
temperature_7obs_max
temperature_7obs_min
temperature_7obs_range
temperature_vs_7obs_mean


In [54]:
# ============================================================
# Build Temporal RF training dataset
# ============================================================

temporal_train_parts = []


for period in TRAIN_PERIODS:

    print(
        "Loading:",
        period
    )

    file_path = (
        FEATURE_DIR
        / f"{period}_features.parquet"
    )


    df = pd.read_parquet(
        file_path,
        columns=(
            [
                "date",
                "serial_number",
                "fail_within_7_days",
            ]
            + TEMPORAL_RF_FEATURES
        )
    )


    df = df.merge(
        TRAIN_SAMPLE_KEYS,
        on=[
            "date",
            "serial_number",
        ],
        how="inner",
        validate="one_to_one"
    )


    temporal_train_parts.append(
        df
    )


train_temporal = pd.concat(
    temporal_train_parts,
    ignore_index=True
)


del temporal_train_parts


print(
    "\nTemporal training rows:",
    f"{len(train_temporal):,}"
)

print(
    "\nClass distribution:"
)

print(
    train_temporal[
        "fail_within_7_days"
    ].value_counts()
)

Loading: 2025_Q2
Loading: 2025_Q3

Temporal training rows: 224,196

Class distribution:
fail_within_7_days
0    213520
1     10676
Name: count, dtype: Int64


In [55]:
# ============================================================
# Train Temporal Random Forest v2
# ============================================================

X_train_temporal = train_temporal[
    TEMPORAL_RF_FEATURES
]


y_train_temporal = (
    train_temporal[
        "fail_within_7_days"
    ]
    .astype(int)
)


temporal_rf_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        ),
    ]
)


print(
    "\n============================================"
)
print(
    "TRAINING TEMPORAL RANDOM FOREST V2"
)
print(
    "============================================"
)


temporal_rf_pipeline.fit(
    X_train_temporal,
    y_train_temporal
)


print(
    "PASS: Temporal RF v2 trained."
)


TRAINING TEMPORAL RANDOM FOREST V2
PASS: Temporal RF v2 trained.


In [56]:
# ============================================================
# Temporal RF v2 validation
# ============================================================

validation_temporal = pd.read_parquet(
    validation_file,
    columns=(
        [
            "serial_number",
            "fail_within_7_days",
        ]
        + TEMPORAL_RF_FEATURES
    )
)


validation_temporal = validation_temporal[
    validation_temporal[
        "fail_within_7_days"
    ].notna()
].copy()


X_validation_temporal = (
    validation_temporal[
        TEMPORAL_RF_FEATURES
    ]
)


validation_probability_temporal = (
    temporal_rf_pipeline
    .predict_proba(
        X_validation_temporal
    )[:, 1]
)


pr_auc_temporal = (
    average_precision_score(
        y_validation,
        validation_probability_temporal
    )
)


roc_auc_temporal = (
    roc_auc_score(
        y_validation,
        validation_probability_temporal
    )
)


print(
    "\n============================================"
)
print(
    "TEMPORAL RF V2 - Q4 VALIDATION"
)
print(
    "============================================"
)

print(
    "PR-AUC:",
    f"{pr_auc_temporal:.6f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_temporal:.6f}"
)

print(
    "PR-AUC change vs Baseline A:",
    f"{pr_auc_temporal - pr_auc_A:+.6f}"
)

print(
    "PR-AUC change vs Baseline B:",
    f"{pr_auc_temporal - pr_auc_B:+.6f}"
)


TEMPORAL RF V2 - Q4 VALIDATION
PR-AUC: 0.025530
ROC-AUC: 0.895347
PR-AUC change vs Baseline A: +0.007921
PR-AUC change vs Baseline B: +0.003930


In [57]:
model_comparison = pd.DataFrame(
    [
        {
            "model":
                "Baseline A - Current SMART",

            "features":
                len(BASELINE_A_FEATURES),

            "pr_auc":
                pr_auc_A,

            "roc_auc":
                roc_auc_A,
        },

        {
            "model":
                "Baseline B - SMART + Indicators",

            "features":
                len(BASELINE_B_FEATURES),

            "pr_auc":
                pr_auc_B,

            "roc_auc":
                roc_auc_B,
        },

        {
            "model":
                "Temporal RF v2",

            "features":
                len(TEMPORAL_RF_FEATURES),

            "pr_auc":
                pr_auc_temporal,

            "roc_auc":
                roc_auc_temporal,
        },
    ]
)


model_comparison[
    "pr_lift_vs_prevalence"
] = (
    model_comparison[
        "pr_auc"
    ]
    / validation_prevalence
)


print(
    "\n============================================"
)
print(
    "MAINTENANCE V2 MODEL COMPARISON"
)
print(
    "============================================"
)


display(
    model_comparison
)


MAINTENANCE V2 MODEL COMPARISON


,model,features,pr_auc,roc_auc,pr_lift_vs_prevalence
0,Baseline A - Current SMART,8,0.017609,0.868802,90.300623
1,Baseline B - SMART + Indicators,12,0.021600,0.870877,110.767498
2,Temporal RF v2,33,0.025530,0.895347,130.923782


In [58]:
# ============================================================
# Temporal RF v2 threshold sensitivity
# ============================================================

TEMPORAL_THRESHOLDS = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.60,
    0.70,
]


temporal_threshold_rows = []


total_failed_drives = (
    validation_temporal.loc[
        y_validation.eq(1),
        "serial_number"
    ]
    .nunique()
)


for threshold in TEMPORAL_THRESHOLDS:

    prediction = (
        validation_probability_temporal
        >= threshold
    ).astype(int)


    precision = precision_score(
        y_validation,
        prediction,
        zero_division=0
    )


    recall = recall_score(
        y_validation,
        prediction,
        zero_division=0
    )


    f1 = f1_score(
        y_validation,
        prediction,
        zero_division=0
    )


    alerts = int(
        prediction.sum()
    )


    detected_failed_drives = (
        validation_temporal.loc[
            (
                prediction == 1
            )
            &
            (
                y_validation.to_numpy()
                == 1
            ),
            "serial_number"
        ]
        .nunique()
    )


    drive_recall = (
        detected_failed_drives
        / total_failed_drives
    )


    alert_rate = (
        alerts
        / len(validation_temporal)
    )


    temporal_threshold_rows.append(
        {
            "threshold":
                threshold,

            "precision":
                precision,

            "row_recall":
                recall,

            "f1":
                f1,

            "alerts":
                alerts,

            "alert_rate_pct":
                alert_rate * 100,

            "alerts_per_1000_drive_days":
                alert_rate * 1000,

            "detected_failed_drives":
                detected_failed_drives,

            "total_failed_drives":
                total_failed_drives,

            "drive_recall":
                drive_recall,
        }
    )


temporal_thresholds = pd.DataFrame(
    temporal_threshold_rows
)


print(
    "\n============================================"
)
print(
    "TEMPORAL RF V2 THRESHOLD ANALYSIS"
)
print(
    "============================================"
)


display(
    temporal_thresholds
)


TEMPORAL RF V2 THRESHOLD ANALYSIS


,threshold,precision,row_recall,f1,alerts,alert_rate_pct,alerts_per_1000_drive_days,detected_failed_drives,total_failed_drives,drive_recall
0,0.05,0.001355,0.762786,0.002706,2068749,10.973984,109.739836,543,631,0.860539
1,0.10,0.002385,0.688792,0.004753,1061660,5.631732,56.317318,500,631,0.792393
2,0.15,0.003385,0.639554,0.006734,694595,3.684581,36.845814,474,631,0.751189
3,0.20,0.004465,0.597116,0.008863,491616,2.607849,26.078494,460,631,0.729002
4,0.25,0.005804,0.562024,0.011489,355987,1.888385,18.883854,446,631,0.706815
5,0.30,0.007463,0.525299,0.014717,258733,1.372487,13.724873,425,631,0.673534
6,0.35,0.009609,0.492111,0.018851,188252,0.998610,9.986104,408,631,0.646593
7,0.40,0.012041,0.454842,0.023460,138864,0.736625,7.366245,393,631,0.622821
8,0.45,0.015127,0.427911,0.029221,103986,0.551609,5.516090,381,631,0.603803
9,0.50,0.018602,0.400435,0.035552,79132,0.419767,4.197673,363,631,0.575277


In [59]:
# ============================================================
# Temporal RF v2 feature importance
# ============================================================

rf_model = (
    temporal_rf_pipeline
    .named_steps["model"]
)


feature_importance = pd.DataFrame(
    {
        "feature":
            TEMPORAL_RF_FEATURES,

        "importance":
            rf_model.feature_importances_,
    }
)


feature_importance = (
    feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


feature_importance[
    "importance_pct"
] = (
    feature_importance[
        "importance"
    ]
    * 100
)


feature_importance[
    "cumulative_importance_pct"
] = (
    feature_importance[
        "importance_pct"
    ]
    .cumsum()
)


print(
    "\n============================================"
)
print(
    "TEMPORAL RF V2 FEATURE IMPORTANCE"
)
print(
    "============================================"
)


display(
    feature_importance
    .head(20)
)


TEMPORAL RF V2 FEATURE IMPORTANCE


,feature,importance,importance_pct,cumulative_importance_pct
0,smart_9_raw,0.093823,9.382327,9.382327
1,smart_198_raw,0.085076,8.507573,17.889900
2,smart_193_raw,0.082890,8.289009,26.178909
3,smart_5_raw_delta_7,0.069585,6.958458,33.137366
4,smart_5_raw,0.056213,5.621334,38.758701
5,smart_198_nonzero,0.055399,5.539904,44.298604
6,smart_192_raw,0.051839,5.183904,49.482508
7,temperature_7obs_mean,0.043861,4.386127,53.868635
8,smart_12_raw,0.038727,3.872715,57.741350
9,smart_4_raw,0.037276,3.727634,61.468984


In [60]:
RESULTS_DIR = (
    PROCESSED_DIR
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


model_comparison.to_csv(
    RESULTS_DIR
    / "maintenance_v2_model_comparison.csv",
    index=False
)


baseline_a_thresholds.to_csv(
    RESULTS_DIR
    / "baseline_a_thresholds.csv",
    index=False
)


temporal_thresholds.to_csv(
    RESULTS_DIR
    / "temporal_rf_v2_thresholds.csv",
    index=False
)


feature_importance.to_csv(
    RESULTS_DIR
    / "temporal_rf_v2_feature_importance.csv",
    index=False
)


print(
    "PASS: Maintenance v2 experimental "
    "results saved."
)

PASS: Maintenance v2 experimental results saved.


In [61]:
# ============================================================
# HistGradientBoosting comparison
# ============================================================

from sklearn.ensemble import (
    HistGradientBoostingClassifier
)


hgb_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.08,
                max_iter=300,
                max_leaf_nodes=31,
                min_samples_leaf=20,
                l2_regularization=1.0,
                random_state=RANDOM_STATE,
            )
        ),
    ]
)


print(
    "\n============================================"
)
print(
    "TRAINING HISTGRADIENTBOOSTING"
)
print(
    "============================================"
)


hgb_pipeline.fit(
    X_train_temporal,
    y_train_temporal
)


print(
    "PASS: HistGradientBoosting trained."
)


TRAINING HISTGRADIENTBOOSTING
PASS: HistGradientBoosting trained.


In [62]:
# ============================================================
# HistGradientBoosting Q4 validation
# ============================================================

validation_probability_hgb = (
    hgb_pipeline
    .predict_proba(
        X_validation_temporal
    )[:, 1]
)


pr_auc_hgb = (
    average_precision_score(
        y_validation,
        validation_probability_hgb
    )
)


roc_auc_hgb = (
    roc_auc_score(
        y_validation,
        validation_probability_hgb
    )
)


print(
    "\n============================================"
)
print(
    "HISTGRADIENTBOOSTING - Q4 VALIDATION"
)
print(
    "============================================"
)

print(
    "PR-AUC:",
    f"{pr_auc_hgb:.6f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_hgb:.6f}"
)

print(
    "PR-AUC change vs Temporal RF:",
    f"{pr_auc_hgb - pr_auc_temporal:+.6f}"
)


HISTGRADIENTBOOSTING - Q4 VALIDATION
PR-AUC: 0.023807
ROC-AUC: 0.902652
PR-AUC change vs Temporal RF: -0.001723


In [63]:
algorithm_comparison = pd.DataFrame(
    [
        {
            "model":
                "Baseline A - Current SMART",
            "pr_auc":
                pr_auc_A,
            "roc_auc":
                roc_auc_A,
        },

        {
            "model":
                "Baseline B - SMART + Indicators",
            "pr_auc":
                pr_auc_B,
            "roc_auc":
                roc_auc_B,
        },

        {
            "model":
                "Temporal Random Forest v2",
            "pr_auc":
                pr_auc_temporal,
            "roc_auc":
                roc_auc_temporal,
        },

        {
            "model":
                "Temporal HistGradientBoosting",
            "pr_auc":
                pr_auc_hgb,
            "roc_auc":
                roc_auc_hgb,
        },
    ]
)


algorithm_comparison[
    "pr_lift_vs_prevalence"
] = (
    algorithm_comparison["pr_auc"]
    / validation_prevalence
)


display(
    algorithm_comparison
)

,model,pr_auc,roc_auc,pr_lift_vs_prevalence
0,Baseline A - Current SMART,0.017609,0.868802,90.300623
1,Baseline B - SMART + Indicators,0.021600,0.870877,110.767498
2,Temporal Random Forest v2,0.025530,0.895347,130.923782
3,Temporal HistGradientBoosting,0.023807,0.902652,122.089292


In [64]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

REPORT_THRESHOLD = 0.60

prediction_temporal = (
    validation_probability_temporal
    >= REPORT_THRESHOLD
).astype(int)


accuracy = accuracy_score(
    y_validation,
    prediction_temporal
)

balanced_accuracy = balanced_accuracy_score(
    y_validation,
    prediction_temporal
)

precision = precision_score(
    y_validation,
    prediction_temporal,
    zero_division=0
)

recall = recall_score(
    y_validation,
    prediction_temporal,
    zero_division=0
)

f1 = f1_score(
    y_validation,
    prediction_temporal,
    zero_division=0
)

cm = confusion_matrix(
    y_validation,
    prediction_temporal
)


print(
    "\n============================================"
)
print(
    "TEMPORAL RF V2 - VALIDATION METRICS"
)
print(
    "============================================"
)

print(
    "Threshold:",
    REPORT_THRESHOLD
)

print(
    "Accuracy:",
    f"{accuracy:.6%}"
)

print(
    "Balanced accuracy:",
    f"{balanced_accuracy:.6%}"
)

print(
    "Precision:",
    f"{precision:.6%}"
)

print(
    "Recall:",
    f"{recall:.6%}"
)

print(
    "F1:",
    f"{f1:.6f}"
)

print(
    "PR-AUC:",
    f"{pr_auc_temporal:.6f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_temporal:.6f}"
)

print(
    "\nConfusion matrix:"
)

print(cm)


TEMPORAL RF V2 - VALIDATION METRICS
Threshold: 0.6
Accuracy: 99.751201%
Balanced accuracy: 66.353824%
Precision: 2.652909%
Recall: 32.943417%
F1: 0.049104
PR-AUC: 0.025530
ROC-AUC: 0.895347

Confusion matrix:
[[18803282    44437]
 [    2465     1211]]


In [66]:
# ============================================================
# Restore drive-model information for Q4 validation
# ============================================================

validation_model_info = pd.read_parquet(
    validation_file,
    columns=[
        "serial_number",
        "model",
        "fail_within_7_days",
    ]
)


validation_model_info = (
    validation_model_info[
        validation_model_info[
            "fail_within_7_days"
        ].notna()
    ]
    .reset_index(
        drop=True
    )
)


validation_temporal = (
    validation_temporal
    .reset_index(
        drop=True
    )
)


# ============================================================
# Safety check
# ============================================================

if (
    len(validation_model_info)
    != len(validation_temporal)
):

    raise AssertionError(
        "Validation row counts do not match."
    )


if not (
    validation_model_info[
        "serial_number"
    ].to_numpy()
    ==
    validation_temporal[
        "serial_number"
    ].to_numpy()
).all():

    raise AssertionError(
        "Validation row order does not match."
    )


# ============================================================
# Add model column
# ============================================================

validation_temporal[
    "model"
] = validation_model_info[
    "model"
].to_numpy()


print(
    "PASS: Drive-model information restored."
)

print(
    "Validation rows:",
    f"{len(validation_temporal):,}"
)

print(
    "Drive models:",
    validation_temporal[
        "model"
    ].nunique()
)

PASS: Drive-model information restored.
Validation rows: 18,851,395
Drive models: 10


In [67]:
# ============================================================
# Temporal RF v2 - per-drive-model Q4 validation
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


per_model_validation = (
    validation_temporal[
        [
            "serial_number",
            "model",
            "fail_within_7_days",
        ]
    ]
    .copy()
)


per_model_validation[
    "failure_probability"
] = validation_probability_temporal


per_model_rows = []


for model, model_df in (
    per_model_validation
    .groupby("model")
):

    y_true = (
        model_df[
            "fail_within_7_days"
        ]
        .astype(int)
    )

    y_score = (
        model_df[
            "failure_probability"
        ]
    )


    positives = int(
        y_true.sum()
    )

    failed_drives = (
        model_df.loc[
            y_true.eq(1),
            "serial_number"
        ]
        .nunique()
    )


    prevalence = (
        y_true.mean()
    )


    # Need both classes for ROC-AUC
    if y_true.nunique() == 2:

        roc_auc = roc_auc_score(
            y_true,
            y_score
        )

        pr_auc = average_precision_score(
            y_true,
            y_score
        )

    else:

        roc_auc = float("nan")
        pr_auc = float("nan")


    pr_lift = (
        pr_auc / prevalence
        if prevalence > 0
        else float("nan")
    )


    per_model_rows.append(
        {
            "model":
                model,

            "drive_days":
                len(model_df),

            "positive_rows":
                positives,

            "failed_drives":
                failed_drives,

            "prevalence_pct":
                prevalence * 100,

            "pr_auc":
                pr_auc,

            "roc_auc":
                roc_auc,

            "pr_lift_vs_prevalence":
                pr_lift,
        }
    )


per_model_metrics = (
    pd.DataFrame(
        per_model_rows
    )
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n============================================"
)
print(
    "TEMPORAL RF V2 - PER-MODEL Q4 PERFORMANCE"
)
print(
    "============================================"
)


display(
    per_model_metrics
)


TEMPORAL RF V2 - PER-MODEL Q4 PERFORMANCE


,model,drive_days,positive_rows,failed_drives,prevalence_pct,pr_auc,roc_auc,pr_lift_vs_prevalence
0,ST24000NM002H,676349,455,68,0.067273,0.094428,0.674180,140.364972
1,ST16000NM001G,3148996,207,35,0.006574,0.054812,0.947733,833.833429
2,ST12000NM0008,1715597,622,109,0.036256,0.053538,0.889212,147.667706
3,HGST HUH721212ALN604,900887,355,68,0.039406,0.025848,0.912367,65.593481
4,TOSHIBA MG07ACA14TA,3428458,448,75,0.013067,0.022946,0.883307,175.602691
5,ST8000NM0055,1217772,174,35,0.014288,0.020373,0.870729,142.585550
6,HGST HUH721212ALE604,1221414,427,75,0.034959,0.016582,0.932197,47.432675
7,WDC WUH721816ALE6L4,2421796,298,46,0.012305,0.008524,0.908747,69.272193
8,TOSHIBA MG08ACA16TA,3670865,448,74,0.012204,0.007336,0.775517,60.107039
9,TOSHIBA MG08ACA16TEY,449261,242,46,0.053866,0.006076,0.747262,11.280394


In [68]:
# ============================================================
# Per-model operating performance
# ============================================================

REFERENCE_THRESHOLD = 0.60


per_model_validation[
    "prediction"
] = (
    per_model_validation[
        "failure_probability"
    ]
    >= REFERENCE_THRESHOLD
).astype(int)


operating_rows = []


for model, model_df in (
    per_model_validation
    .groupby("model")
):

    y_true = (
        model_df[
            "fail_within_7_days"
        ]
        .astype(int)
    )

    y_pred = (
        model_df[
            "prediction"
        ]
    )


    actual_failed_drives = (
        model_df.loc[
            y_true.eq(1),
            "serial_number"
        ]
        .nunique()
    )


    detected_failed_drives = (
        model_df.loc[
            y_true.eq(1)
            &
            y_pred.eq(1),
            "serial_number"
        ]
        .nunique()
    )


    drive_recall = (
        detected_failed_drives
        / actual_failed_drives
        if actual_failed_drives > 0
        else float("nan")
    )


    alerts = int(
        y_pred.sum()
    )


    alerts_per_1000 = (
        alerts
        / len(model_df)
        * 1000
    )


    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )


    row_recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )


    operating_rows.append(
        {
            "model":
                model,

            "failed_drives":
                actual_failed_drives,

            "detected_failed_drives":
                detected_failed_drives,

            "drive_recall":
                drive_recall,

            "precision":
                precision,

            "row_recall":
                row_recall,

            "alerts":
                alerts,

            "alerts_per_1000_drive_days":
                alerts_per_1000,
        }
    )


per_model_operating = (
    pd.DataFrame(
        operating_rows
    )
    .sort_values(
        "drive_recall",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n============================================"
)
print(
    "PER-MODEL PERFORMANCE @ THRESHOLD 0.60"
)
print(
    "============================================"
)


display(
    per_model_operating
)


PER-MODEL PERFORMANCE @ THRESHOLD 0.60


,model,failed_drives,detected_failed_drives,drive_recall,precision,row_recall,alerts,alerts_per_1000_drive_days
0,ST12000NM0008,109,75,0.688073,0.036104,0.467846,8060,4.698073
1,ST16000NM001G,35,24,0.685714,0.027367,0.589372,4458,1.415689
2,HGST HUH721212ALN604,68,42,0.617647,0.027078,0.363380,4764,5.288122
3,ST8000NM0055,35,21,0.600000,0.017481,0.281609,2803,2.301744
4,HGST HUH721212ALE604,75,39,0.520000,0.021925,0.318501,6203,5.078540
5,TOSHIBA MG07ACA14TA,75,38,0.506667,0.032984,0.379464,5154,1.503300
6,TOSHIBA MG08ACA16TA,74,29,0.391892,0.015347,0.241071,7037,1.916987
7,ST24000NM002H,68,22,0.323529,0.098637,0.270330,1247,1.843723
8,TOSHIBA MG08ACA16TEY,46,14,0.304348,0.010602,0.165289,3773,8.398236
9,WDC WUH721816ALE6L4,46,12,0.260870,0.020009,0.144295,2149,0.887358


In [69]:
# ============================================================
# Hardware-Aware Temporal RF v2.1
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier


NUMERIC_FEATURES = (
    TEMPORAL_RF_FEATURES
)

CATEGORICAL_FEATURES = [
    "model"
]


hardware_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(
                strategy="median"
            ),
            NUMERIC_FEATURES
        ),

        (
            "model",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            CATEGORICAL_FEATURES
        ),
    ]
)


hardware_rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            hardware_preprocessor
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        ),
    ]
)

In [70]:
# ============================================================
# Build hardware-aware training dataset
# ============================================================

hardware_train_parts = []


for period in TRAIN_PERIODS:

    print(
        "Loading:",
        period
    )

    file_path = (
        FEATURE_DIR
        / f"{period}_features.parquet"
    )


    df = pd.read_parquet(
        file_path,
        columns=(
            [
                "date",
                "serial_number",
                "model",
                "fail_within_7_days",
            ]
            + TEMPORAL_RF_FEATURES
        )
    )


    df = df.merge(
        TRAIN_SAMPLE_KEYS,
        on=[
            "date",
            "serial_number",
        ],
        how="inner",
        validate="one_to_one"
    )


    hardware_train_parts.append(
        df
    )


train_hardware = pd.concat(
    hardware_train_parts,
    ignore_index=True
)


del hardware_train_parts


print(
    "\n============================================"
)
print(
    "HARDWARE-AWARE TRAINING DATA"
)
print(
    "============================================"
)

print(
    "Rows:",
    f"{len(train_hardware):,}"
)

print(
    "Drive models:",
    train_hardware[
        "model"
    ].nunique()
)

print(
    "\nClass distribution:"
)

print(
    train_hardware[
        "fail_within_7_days"
    ].value_counts()
)

Loading: 2025_Q2
Loading: 2025_Q3

HARDWARE-AWARE TRAINING DATA
Rows: 224,196
Drive models: 10

Class distribution:
fail_within_7_days
0    213520
1     10676
Name: count, dtype: Int64


In [71]:
# ============================================================
# Train Hardware-Aware Temporal RF v2.1
# ============================================================

X_train_hardware = train_hardware[
    [
        "model"
    ]
    + TEMPORAL_RF_FEATURES
]


y_train_hardware = (
    train_hardware[
        "fail_within_7_days"
    ]
    .astype(int)
)


print(
    "\n============================================"
)
print(
    "TRAINING HARDWARE-AWARE TEMPORAL RF V2.1"
)
print(
    "============================================"
)

print(
    "Training rows:",
    f"{len(X_train_hardware):,}"
)

print(
    "Numeric features:",
    len(TEMPORAL_RF_FEATURES)
)

print(
    "Categorical features:",
    1
)


hardware_rf_pipeline.fit(
    X_train_hardware,
    y_train_hardware
)


print(
    "\nPASS: Hardware-Aware "
    "Temporal RF v2.1 trained."
)


TRAINING HARDWARE-AWARE TEMPORAL RF V2.1
Training rows: 224,196
Numeric features: 33
Categorical features: 1

PASS: Hardware-Aware Temporal RF v2.1 trained.


In [72]:
# ============================================================
# Hardware-Aware Temporal RF v2.1 - Q4 validation
# ============================================================

validation_hardware = pd.read_parquet(
    validation_file,
    columns=(
        [
            "serial_number",
            "model",
            "fail_within_7_days",
        ]
        + TEMPORAL_RF_FEATURES
    )
)


validation_hardware = (
    validation_hardware[
        validation_hardware[
            "fail_within_7_days"
        ].notna()
    ]
    .copy()
)


X_validation_hardware = (
    validation_hardware[
        [
            "model"
        ]
        + TEMPORAL_RF_FEATURES
    ]
)


y_validation_hardware = (
    validation_hardware[
        "fail_within_7_days"
    ]
    .astype(int)
)


# Safety check
if not (
    y_validation_hardware.to_numpy()
    ==
    y_validation.to_numpy()
).all():

    raise AssertionError(
        "Validation labels do not match "
        "previous experiments."
    )


validation_probability_hardware = (
    hardware_rf_pipeline
    .predict_proba(
        X_validation_hardware
    )[:, 1]
)


pr_auc_hardware = (
    average_precision_score(
        y_validation_hardware,
        validation_probability_hardware
    )
)


roc_auc_hardware = (
    roc_auc_score(
        y_validation_hardware,
        validation_probability_hardware
    )
)


print(
    "\n============================================"
)
print(
    "HARDWARE-AWARE TEMPORAL RF V2.1"
)
print(
    "Q4 VALIDATION"
)
print(
    "============================================"
)

print(
    "PR-AUC:",
    f"{pr_auc_hardware:.6f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_hardware:.6f}"
)

print(
    "PR-AUC change vs Temporal RF v2:",
    f"{pr_auc_hardware - pr_auc_temporal:+.6f}"
)


HARDWARE-AWARE TEMPORAL RF V2.1
Q4 VALIDATION
PR-AUC: 0.025340
ROC-AUC: 0.894235
PR-AUC change vs Temporal RF v2: -0.000190


In [73]:
hardware_comparison = pd.DataFrame(
    [
        {
            "model":
                "Baseline A - Current SMART",
            "pr_auc":
                pr_auc_A,
            "roc_auc":
                roc_auc_A,
        },

        {
            "model":
                "Baseline B - SMART + Indicators",
            "pr_auc":
                pr_auc_B,
            "roc_auc":
                roc_auc_B,
        },

        {
            "model":
                "Temporal RF v2",
            "pr_auc":
                pr_auc_temporal,
            "roc_auc":
                roc_auc_temporal,
        },

        {
            "model":
                "Temporal HGB",
            "pr_auc":
                pr_auc_hgb,
            "roc_auc":
                roc_auc_hgb,
        },

        {
            "model":
                "Hardware-Aware Temporal RF v2.1",
            "pr_auc":
                pr_auc_hardware,
            "roc_auc":
                roc_auc_hardware,
        },
    ]
)


hardware_comparison[
    "pr_lift_vs_prevalence"
] = (
    hardware_comparison[
        "pr_auc"
    ]
    / validation_prevalence
)


print(
    "\n============================================"
)
print(
    "MAINTENANCE V2 MODEL COMPARISON"
)
print(
    "============================================"
)


display(
    hardware_comparison
)


MAINTENANCE V2 MODEL COMPARISON


,model,pr_auc,roc_auc,pr_lift_vs_prevalence
0,Baseline A - Current SMART,0.017609,0.868802,90.300623
1,Baseline B - SMART + Indicators,0.021600,0.870877,110.767498
2,Temporal RF v2,0.025530,0.895347,130.923782
3,Temporal HGB,0.023807,0.902652,122.089292
4,Hardware-Aware Temporal RF v2.1,0.025340,0.894235,129.946991


### Hardware-Identity Ablation

Adding drive-model identity as a one-hot encoded categorical feature did
not improve Q4 validation performance.

- Temporal RF v2 PR-AUC: 0.025530
- Hardware-aware Temporal RF v2.1 PR-AUC: 0.025340
- Temporal RF v2 ROC-AUC: 0.895347
- Hardware-aware Temporal RF v2.1 ROC-AUC: 0.894235

The hardware-aware variant was therefore not selected. The generalized
model continues to use common SMART health and temporal features without
explicit drive-model identity.

In [74]:
n_estimators=300
max_depth=None
min_samples_leaf=2
max_features="sqrt"

In [75]:
# ============================================================
# Small Random Forest tuning experiment
# ============================================================

RF_CONFIGS = [
    {
        "name": "RF_CURRENT",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },

    {
        "name": "RF_LEAF_5",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },

    {
        "name": "RF_LEAF_10",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
    },

    {
        "name": "RF_DEPTH_20",
        "n_estimators": 300,
        "max_depth": 20,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },

    {
        "name": "RF_FEATURES_0.5",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": 0.5,
    },
]

In [76]:
rf_tuning_rows = []

rf_tuning_models = {}


for config in RF_CONFIGS:

    print(
        "\n============================================"
    )
    print(
        "TRAINING:",
        config["name"]
    )
    print(
        "============================================"
    )


    pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "model",
                RandomForestClassifier(
                    n_estimators=
                        config["n_estimators"],

                    max_depth=
                        config["max_depth"],

                    min_samples_leaf=
                        config["min_samples_leaf"],

                    max_features=
                        config["max_features"],

                    random_state=
                        RANDOM_STATE,

                    n_jobs=-1,
                )
            ),
        ]
    )


    pipeline.fit(
        X_train_temporal,
        y_train_temporal
    )


    probability = (
        pipeline
        .predict_proba(
            X_validation_temporal
        )[:, 1]
    )


    pr_auc = average_precision_score(
        y_validation,
        probability
    )


    roc_auc = roc_auc_score(
        y_validation,
        probability
    )


    rf_tuning_rows.append(
        {
            "configuration":
                config["name"],

            "pr_auc":
                pr_auc,

            "roc_auc":
                roc_auc,

            "pr_lift":
                pr_auc
                / validation_prevalence,
        }
    )


    rf_tuning_models[
        config["name"]
    ] = pipeline


rf_tuning_results = (
    pd.DataFrame(
        rf_tuning_rows
    )
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n============================================"
)
print(
    "RF TUNING RESULTS"
)
print(
    "============================================"
)


display(
    rf_tuning_results
)


TRAINING: RF_CURRENT

TRAINING: RF_LEAF_5

TRAINING: RF_LEAF_10

TRAINING: RF_DEPTH_20

TRAINING: RF_FEATURES_0.5

RF TUNING RESULTS


,configuration,pr_auc,roc_auc,pr_lift
0,RF_CURRENT,0.025530,0.895347,130.923920
1,RF_LEAF_10,0.025289,0.900233,129.688105
2,RF_LEAF_5,0.025211,0.897759,129.286108
3,RF_DEPTH_20,0.024691,0.903073,126.620867
4,RF_FEATURES_0.5,0.023327,0.887630,119.625978


In [77]:
# ============================================================
# 50:1 negative sampling experiment
# ============================================================

RATIO_50 = 50

target_negative_50 = (
    total_positive
    * RATIO_50
)

print(
    "Positive rows:",
    f"{total_positive:,}"
)

print(
    "Target negatives:",
    f"{target_negative_50:,}"
)

Positive rows: 10,676
Target negatives: 533,800


In [78]:
negative_50_parts = []


for period in TRAIN_PERIODS:

    print(
        "Loading:",
        period
    )

    file_path = (
        FEATURE_DIR
        / f"{period}_features.parquet"
    )


    df = pd.read_parquet(
        file_path,
        columns=(
            [
                "date",
                "serial_number",
                "fail_within_7_days",
            ]
            + TEMPORAL_RF_FEATURES
        )
    )


    df = df[
        df[
            "fail_within_7_days"
        ].eq(0)
    ]


    proportion = (
        len(df)
        / (
            integrity_summary.loc[
                integrity_summary[
                    "period"
                ].isin(
                    TRAIN_PERIODS
                ),
                "predictive_rows"
            ].sum()
            -
            total_positive
        )
    )


    sample_size = int(
        round(
            target_negative_50
            * proportion
        )
    )


    sampled = df.sample(
        n=min(
            sample_size,
            len(df)
        ),
        random_state=RANDOM_STATE
    )


    negative_50_parts.append(
        sampled
    )


    del df

Loading: 2025_Q2
Loading: 2025_Q3


In [79]:
negative_50 = pd.concat(
    negative_50_parts,
    ignore_index=True
)


# Need the 33-feature positive observations
positive_temporal_parts = []


for period in TRAIN_PERIODS:

    file_path = (
        FEATURE_DIR
        / f"{period}_features.parquet"
    )


    positive = pd.read_parquet(
        file_path,
        columns=(
            [
                "date",
                "serial_number",
                "fail_within_7_days",
            ]
            + TEMPORAL_RF_FEATURES
        )
    )


    positive = positive[
        positive[
            "fail_within_7_days"
        ].eq(1)
    ]


    positive_temporal_parts.append(
        positive
    )


positive_temporal = pd.concat(
    positive_temporal_parts,
    ignore_index=True
)


train_50 = pd.concat(
    [
        positive_temporal,
        negative_50,
    ],
    ignore_index=True
)


train_50 = (
    train_50
    .sample(
        frac=1,
        random_state=RANDOM_STATE
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n============================================"
)
print(
    "50:1 TRAINING SAMPLE"
)
print(
    "============================================"
)

print(
    train_50[
        "fail_within_7_days"
    ].value_counts()
)


50:1 TRAINING SAMPLE
fail_within_7_days
0    533800
1     10676
Name: count, dtype: Int64


In [80]:
X_train_50 = train_50[
    TEMPORAL_RF_FEATURES
]

y_train_50 = (
    train_50[
        "fail_within_7_days"
    ]
    .astype(int)
)


rf_50_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        ),
    ]
)


print(
    "Training RF 50:1..."
)

rf_50_pipeline.fit(
    X_train_50,
    y_train_50
)


validation_probability_50 = (
    rf_50_pipeline
    .predict_proba(
        X_validation_temporal
    )[:, 1]
)


pr_auc_50 = average_precision_score(
    y_validation,
    validation_probability_50
)

roc_auc_50 = roc_auc_score(
    y_validation,
    validation_probability_50
)


print(
    "\n============================================"
)
print(
    "RF 50:1 - Q4 VALIDATION"
)
print(
    "============================================"
)

print(
    "PR-AUC:",
    f"{pr_auc_50:.6f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_50:.6f}"
)

print(
    "PR-AUC change vs 20:1:",
    f"{pr_auc_50 - pr_auc_temporal:+.6f}"
)

Training RF 50:1...

RF 50:1 - Q4 VALIDATION
PR-AUC: 0.028492
ROC-AUC: 0.891983
PR-AUC change vs 20:1: +0.002962


In [82]:
# ============================================================
# RF 50:1 threshold analysis
# ============================================================

THRESHOLDS_50 = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
]


# ============================================================
# Reset validation alignment
# ============================================================

validation_50 = (
    validation_temporal
    .reset_index(
        drop=True
    )
    .copy()
)


y_validation_50 = (
    validation_50[
        "fail_within_7_days"
    ]
    .astype(int)
    .reset_index(
        drop=True
    )
)


# ============================================================
# Probability alignment check
# ============================================================

if (
    len(validation_probability_50)
    != len(validation_50)
):

    raise AssertionError(
        "Probability and validation row "
        "counts do not match."
    )


print(
    "Validation rows:",
    f"{len(validation_50):,}"
)

print(
    "Probability rows:",
    f"{len(validation_probability_50):,}"
)


# ============================================================
# Failed-drive population
# ============================================================

positive_mask = (
    y_validation_50
    .to_numpy()
    == 1
)


total_failed_drives = (
    validation_50.loc[
        positive_mask,
        "serial_number"
    ]
    .nunique()
)


print(
    "Validation failed drives:",
    total_failed_drives
)


# ============================================================
# Threshold evaluation
# ============================================================

threshold_50_rows = []


for threshold in THRESHOLDS_50:

    prediction = (
        validation_probability_50
        >= threshold
    ).astype(int)


    precision = precision_score(
        y_validation_50,
        prediction,
        zero_division=0
    )


    row_recall = recall_score(
        y_validation_50,
        prediction,
        zero_division=0
    )


    f1 = f1_score(
        y_validation_50,
        prediction,
        zero_division=0
    )


    alerts = int(
        prediction.sum()
    )


    # --------------------------------------------------------
    # Drive-level detection
    # --------------------------------------------------------

    detected_mask = (
        (prediction == 1)
        &
        positive_mask
    )


    detected_failed_drives = (
        validation_50.loc[
            detected_mask,
            "serial_number"
        ]
        .nunique()
    )


    drive_recall = (
        detected_failed_drives
        / total_failed_drives
    )


    # --------------------------------------------------------
    # Alert burden
    # --------------------------------------------------------

    alert_rate = (
        alerts
        / len(validation_50)
    )


    threshold_50_rows.append(
        {
            "threshold":
                threshold,

            "precision":
                precision,

            "row_recall":
                row_recall,

            "f1":
                f1,

            "alerts":
                alerts,

            "alert_rate_pct":
                alert_rate * 100,

            "alerts_per_1000_drive_days":
                alert_rate * 1000,

            "detected_failed_drives":
                detected_failed_drives,

            "total_failed_drives":
                total_failed_drives,

            "drive_recall":
                drive_recall,
        }
    )


# ============================================================
# Results
# ============================================================

rf_50_thresholds = pd.DataFrame(
    threshold_50_rows
)


print(
    "\n============================================"
)
print(
    "RF 50:1 THRESHOLD ANALYSIS"
)
print(
    "============================================"
)


display(
    rf_50_thresholds
)

Validation rows: 18,851,395
Probability rows: 18,851,395
Validation failed drives: 631

RF 50:1 THRESHOLD ANALYSIS


,threshold,precision,row_recall,f1,alerts,alert_rate_pct,alerts_per_1000_drive_days,detected_failed_drives,total_failed_drives,drive_recall
0,0.05,0.002386,0.676823,0.004755,1042802,5.531697,55.316967,500,631,0.792393
1,0.10,0.004256,0.594396,0.008451,513430,2.723565,27.235650,461,631,0.730586
2,0.15,0.006528,0.540261,0.012900,304231,1.613838,16.138381,436,631,0.690967
3,0.20,0.009312,0.495919,0.018280,195777,1.038528,10.385279,417,631,0.660856
4,0.25,0.012507,0.451850,0.024341,132802,0.704468,7.044678,396,631,0.627575
5,0.30,0.016270,0.417301,0.031319,94285,0.500149,5.001487,373,631,0.591125
6,0.35,0.020366,0.381665,0.038668,68891,0.365442,3.654425,357,631,0.565769
7,0.40,0.025241,0.349021,0.047077,50831,0.269641,2.696405,343,631,0.543582
8,0.45,0.030167,0.311480,0.055006,37956,0.201343,2.013432,320,631,0.507132
9,0.50,0.036084,0.278564,0.063892,28378,0.150535,1.505353,296,631,0.469097


In [83]:
# ============================================================
# Warning lead-time analysis
# RF 50:1 / threshold 0.45
# ============================================================

SELECTED_THRESHOLD = 0.45


lead_time_df = validation_50[
    [
        "serial_number",
        "fail_within_7_days",
    ]
].copy()


# We need dates and failure dates.
validation_dates = pd.read_parquet(
    validation_file,
    columns=[
        "date",
        "serial_number",
        "failure_date",
        "fail_within_7_days",
    ]
)


validation_dates = (
    validation_dates[
        validation_dates[
            "fail_within_7_days"
        ].notna()
    ]
    .reset_index(
        drop=True
    )
)


# Safety checks
if len(validation_dates) != len(lead_time_df):

    raise AssertionError(
        "Validation row counts do not match."
    )


if not (
    validation_dates[
        "serial_number"
    ].to_numpy()
    ==
    lead_time_df[
        "serial_number"
    ].to_numpy()
).all():

    raise AssertionError(
        "Validation row order does not match."
    )


lead_time_df[
    "date"
] = pd.to_datetime(
    validation_dates[
        "date"
    ]
)


lead_time_df[
    "failure_date"
] = pd.to_datetime(
    validation_dates[
        "failure_date"
    ]
)


lead_time_df[
    "failure_probability"
] = validation_probability_50


lead_time_df[
    "alert"
] = (
    lead_time_df[
        "failure_probability"
    ]
    >= SELECTED_THRESHOLD
)


# ============================================================
# Failed drives only
# ============================================================

failed_q4 = lead_time_df[
    lead_time_df[
        "fail_within_7_days"
    ].eq(1)
].copy()


# Only actual positive alerts
detected_positive = failed_q4[
    failed_q4[
        "alert"
    ]
].copy()


detected_positive[
    "lead_time_days"
] = (
    detected_positive[
        "failure_date"
    ]
    -
    detected_positive[
        "date"
    ]
).dt.days


# ============================================================
# First warning per failed drive
#
# Earliest alert gives maximum warning time.
# ============================================================

first_warning = (
    detected_positive
    .sort_values(
        [
            "serial_number",
            "date",
        ]
    )
    .groupby(
        "serial_number",
        as_index=False
    )
    .first()
)


print(
    "\n============================================"
)
print(
    "RF 50:1 WARNING LEAD TIME"
)
print(
    "============================================"
)

print(
    "Threshold:",
    SELECTED_THRESHOLD
)

print(
    "Detected failed drives:",
    f"{len(first_warning):,}"
)

print(
    "Median first-warning lead time:",
    f"{first_warning['lead_time_days'].median():.1f}",
    "days"
)

print(
    "Mean first-warning lead time:",
    f"{first_warning['lead_time_days'].mean():.2f}",
    "days"
)

print(
    "Minimum lead time:",
    int(
        first_warning[
            "lead_time_days"
        ].min()
    ),
    "days"
)

print(
    "Maximum lead time:",
    int(
        first_warning[
            "lead_time_days"
        ].max()
    ),
    "days"
)


RF 50:1 WARNING LEAD TIME
Threshold: 0.45
Detected failed drives: 320
Median first-warning lead time: 6.0 days
Mean first-warning lead time: 4.97 days
Minimum lead time: 1 days
Maximum lead time: 7 days


In [84]:
lead_time_distribution = (
    first_warning[
        "lead_time_days"
    ]
    .value_counts()
    .sort_index(
        ascending=False
    )
    .rename_axis(
        "lead_time_days"
    )
    .reset_index(
        name="failed_drives"
    )
)


print(
    "\n============================================"
)
print(
    "FIRST WARNING DISTRIBUTION"
)
print(
    "============================================"
)


display(
    lead_time_distribution
)


FIRST WARNING DISTRIBUTION


,lead_time_days,failed_drives
0,7,141
1,6,33
2,5,26
3,4,27
4,3,25
5,2,23
6,1,45


In [85]:
def evaluate_lead_time(
    threshold,
    validation_df,
    probabilities
):

    df = validation_df.copy()

    df["failure_probability"] = probabilities

    df["alert"] = (
        df["failure_probability"]
        >= threshold
    )

    positive = df[
        df["fail_within_7_days"].eq(1)
    ].copy()

    detected = positive[
        positive["alert"]
    ].copy()

    detected["lead_time_days"] = (
        detected["failure_date"]
        - detected["date"]
    ).dt.days

    first_warning = (
        detected
        .sort_values(
            [
                "serial_number",
                "date"
            ]
        )
        .groupby(
            "serial_number",
            as_index=False
        )
        .first()
    )

    return {
        "threshold":
            threshold,

        "detected_failed_drives":
            first_warning[
                "serial_number"
            ].nunique(),

        "median_lead_time_days":
            first_warning[
                "lead_time_days"
            ].median(),

        "mean_lead_time_days":
            first_warning[
                "lead_time_days"
            ].mean(),

        "seven_day_warnings":
            int(
                first_warning[
                    "lead_time_days"
                ].eq(7)
                .sum()
            ),
    }

In [86]:
lead_time_comparison = pd.DataFrame(
    [
        evaluate_lead_time(
            threshold,
            lead_time_df,
            validation_probability_50
        )
        for threshold in [
            0.40,
            0.45,
            0.50,
        ]
    ]
)

display(
    lead_time_comparison
)

,threshold,detected_failed_drives,median_lead_time_days,mean_lead_time_days,seven_day_warnings
0,0.40,343,6.0,5.078717,157
1,0.45,320,6.0,4.965625,141
2,0.50,296,6.0,4.871622,123


In [87]:
lead_time_df["date"] = pd.to_datetime(
    lead_time_df["date"]
)

lead_time_df["failure_date"] = pd.to_datetime(
    lead_time_df["failure_date"]
)

## Frozen Classical Candidate — Temporal Random Forest v2

Following controlled Q4 validation experiments, the classical
predictive-maintenance candidate is frozen as follows:

### Training
- Training periods: 2025 Q2 + Q3
- Validation period: 2025 Q4
- Final untouched test period: 2026 Q1
- Negative-to-positive training ratio: 50:1

### Model
- RandomForestClassifier
- n_estimators = 300
- max_depth = None
- min_samples_leaf = 2
- max_features = "sqrt"
- random_state = 42

### Features
33 common SMART health and temporal features.

Explicit drive-model identity was evaluated but not retained because
it did not improve validation PR-AUC.

### Candidate operating threshold
0.45

### Q4 validation
- PR-AUC: 0.028492
- ROC-AUC: 0.891983
- Drive-level recall: 50.71%
- Failed drives detected: 320 / 631
- Precision: 3.02%
- Row recall: 31.15%
- Alerts per 1,000 drive-days: 2.01
- Median first-warning lead time: 6 days
- Mean first-warning lead time: 4.97 days
- 141 detected drives received their first warning at the maximum
  7-day prediction horizon.

Overall accuracy is not used as the primary model-selection metric
because of extreme class imbalance.

In [1]:
import sys

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

Python version:
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

Python executable:
c:\Users\Aljawharah\Documents\DC-Guardian\.venv-gru\Scripts\python.exe


In [2]:
import sys
import torch

print(
    "\n============================================"
)
print(
    "DC-GUARDIAN GRU ENVIRONMENT"
)
print(
    "============================================"
)

print(
    "Python:",
    sys.version
)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

print(
    "PyTorch CUDA:",
    torch.version.cuda
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )


DC-GUARDIAN GRU ENVIRONMENT
Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.14.0+cpu
CUDA available: False
PyTorch CUDA: None


In [89]:
GRU_SMART_FEATURES = [
    "smart_5_raw",
    "smart_9_raw",
    "smart_192_raw",
    "smart_193_raw",
    "smart_194_raw",
    "smart_198_raw",
    "smart_4_raw",
    "smart_12_raw",
]


GRU_SEQUENCE_LENGTH = 30

GRU_FAILURE_HORIZON = 7


print(
    "\n============================================"
)
print(
    "GRU EXPERIMENT CONFIGURATION"
)
print(
    "============================================"
)

print(
    "SMART signals:",
    len(GRU_SMART_FEATURES)
)

print(
    "Sequence length:",
    GRU_SEQUENCE_LENGTH
)

print(
    "Prediction horizon:",
    GRU_FAILURE_HORIZON
)


GRU EXPERIMENT CONFIGURATION
SMART signals: 8
Sequence length: 30
Prediction horizon: 7


### GRU Sequence-Model Challenger

A gated recurrent unit (GRU) sequence model was evaluated as a
deep-learning challenger to the frozen Temporal Random Forest.

The GRU consumed 30 consecutive daily observations across eight
common SMART telemetry channels and predicted explicit drive failure
within the following seven days.

The model was trained using 2025 Q2 and Q3 and selected using 2025 Q4.
The 2026 Q1 final-test period remained untouched.

For a fair comparison, both the GRU and frozen Random Forest were
evaluated on the identical GRU-eligible Q4 population.

| Model | PR-AUC | ROC-AUC | PR lift |
|---|---:|---:|---:|
| Temporal RF v2 | 0.022843 | 0.879989 | 133.57x |
| GRU v1 | 0.015752 | 0.869368 | 92.11x |

The GRU reduced PR-AUC by 0.007091 and ROC-AUC by 0.010621 relative
to the Temporal Random Forest on the same evaluation population.

The GRU was therefore retained as an evaluated experimental
challenger but was not promoted as the final predictive-maintenance
candidate.